In [11]:
#!/usr/bin/env python
# coding: utf-8

# ==============================================================
# NOTEBOOK PIETRO — RICONCILIAZIONE FATTURE
# ==============================================================
# Confronto fra le righe delle fatture elettroniche (XML FPR12) e
# le righe dei carichi a magazzino, per far emergere le differenze
# di quantità, prezzo e totale articolo per articolo.
#
#   [1]  Configurazione e sorgenti
#   [2]  Lettura degli XML dallo ZIP su OneLake
#   [3]  Esplosione delle righe fattura e dei codici articolo
#   [4]  Funzioni di normalizzazione
#   [5]  Dizionari e risoluzione delle regole
#   [6]  Diagnostica del dizionario (BLOCCO RIMUOVIBILE)
#   [7]  Carichi: lettura, tipi e filtro sul tipo documento
#   [8]  Carichi: perimetro XML e indice progressivo
#   [9]  Carichi: anagrafiche e sostituzione di CALA
#   [10] Carichi: chiave DDT normalizzata
#   [11] XML: fornitore e chiave DDT normalizzata
#   [12] XML: codici articolo normalizzati
#   [13] XML: righe aggregate (xml_righe_(3))
#   [14] Abbinamento DDT fra XML e carichi (map_ddt)
#   [15] Merge: full outer fra XML e carichi
#   [16] Esportazione
#
# Le celle [2] e [3] alimentano df_xml_fatture e
# df_xml_codici_articolo. Le celle da [7] in poi alimentano
# df_fatture (lato carichi). La cella [8] tiene insieme i due
# lati: è lì che il perimetro viene definito.
#
# Le celle [11] e [12] portano il CFOR sul lato XML e vi
# applicano il dizionario: dipendono da [8], che definisce il
# perimetro, e dalle celle [4] e [5], che espongono le regole.
#
# La cella [13] porta il lato XML alla granularità dei carichi,
# la [14] costruisce il ponte fra i DDT dei due lati e la [15]
# esegue il full outer. Senza [15] non esiste ancora df_merge.
# ==============================================================

# %% [1]


import os
import unicodedata
import zipfile
import xml.etree.ElementTree as ET

import pandas as pd


# ==============================================================
# CONFIGURAZIONE
# ==============================================================
# Unico punto in cui si indicano le sorgenti.
#
# La sezione è volutamente separata dal resto: quando i dati
# arriveranno direttamente dal database basterà agire qui,
# senza toccare le celle successive.
#
#     <cartella>/xml.zip                   XML delle fatture
#     <cartella>/TAB_CM_Fatture_BI.xlsx    carichi
#     <cartella>/ANAGRAFICA_PDV.xlsx       punti di consegna
#     <cartella>/TAB_ANAGRAFICA_EAN.xlsx   anagrafica EAN
#     <cartella>/TAB_ANAGRAFICA_CARF.xlsx  anagrafica codici fornitore
#     <cartella>/Dizionario codici.xlsx    dizionario delle regole
# ==============================================================

cartella = (
    "abfss://d072e408-7302-41de-a976-f6b6b626177e"
    "@onelake.dfs.fabric.microsoft.com/"
    "401c1a6a-fa86-43ab-a153-01e4fa9de79f/Files"
)


# --------------------------------------------------------------
# SORGENTE DEI CARICHI
# --------------------------------------------------------------
# "excel"  legge il file Excel derivato dal database
# "table"  leggerà direttamente la tabella su Fabric
SOURCE_TYPE = "excel"

CARICHI_FILE = "TAB_CM_Fatture_BI.xlsx"
CARICHI_PATH = f"{cartella}/{CARICHI_FILE}"
CARICHI_SHEET = "TAB_CM_Fatture_BI"

# Configurazione futura
TABLE_NAME = "TAB_CM_Fatture_BI"

# Tipi di documento che entrano nel perimetro:
#
#     60I   fattura
#     60C   nota di credito
#
# L'elenco è volutamente una lista: aggiungere una sigla significa
# scriverla qui e nient'altro.
CDOCIAA_FILTRO = ["60I", "60C"]


# --------------------------------------------------------------
# SORGENTE DEGLI XML
# --------------------------------------------------------------

# ZIP presente nella cartella OneLake
zip_path = f"{cartella}/xml.zip"

# Copia locale temporanea dello ZIP
zip_locale_fs = "file:/tmp/xml.zip"
zip_locale_python = "/tmp/xml.zip"


# --------------------------------------------------------------
# DIZIONARI
# --------------------------------------------------------------

percorso_dizionari = f"{cartella}/Dizionario codici.xlsx"

# Nomi dei fogli: modificare qui se cambiano
NOME_FOGLIO_CODICI_XML = "RegoleCodiciXML"
NOME_FOGLIO_CODICI_CARICO = "RegoleCodiciCarico"
NOME_FOGLIO_DDT = "RegoleDDT"

# Copia locale temporanea
dizionari_fs = "file:/tmp/Dizionario codici.xlsx"
dizionari_locale = "/tmp/Dizionario codici.xlsx"


# --------------------------------------------------------------
# ANAGRAFICHE
# --------------------------------------------------------------

CPCO_FILE = "ANAGRAFICA_PDV.xlsx"
CPCO_PATH = f"{cartella}/{CPCO_FILE}"
CPCO_SHEET = "List1"

EAN_FILE = "TAB_ANAGRAFICA_EAN.xlsx"
EAN_PATH = f"{cartella}/{EAN_FILE}"
EAN_SHEET = "TAB_CM_ANAGRAFICA_EAN_BI"

CARF_FILE = "TAB_ANAGRAFICA_CARF.xlsx"
CARF_PATH = f"{cartella}/{CARF_FILE}"
CARF_SHEET = "TAB_CM_ANAGRAFICA_CARF_BI"
# In alternativa, per aprire il primo foglio disponibile:
# CARF_SHEET = 0


# %% [2]


# ==============================================================
# LETTURA DEGLI XML DALLO ZIP
# ==============================================================
# I percorsi dello ZIP e della copia locale sono definiti
# nella cella [1]: qui vengono soltanto utilizzati.
# ==============================================================

denominazioni = []
errori = []

# Contiene gli XML FPR12 senza allegati
xml_in_memoria = []

# Elenco dei file XML presenti nello ZIP.
# Viene valorizzata dopo la lettura dell'archivio.
# L'inizializzazione anticipata evita che il riepilogo finale
# sollevi un NameError quando l'apertura dello ZIP non riesce.
file_xml = []


try:
    # Elimina l'eventuale copia locale di una precedente esecuzione
    if notebookutils.fs.exists(zip_locale_fs):
        notebookutils.fs.rm(zip_locale_fs)

    # Copia lo ZIP da OneLake al filesystem locale del notebook
    copia_eseguita = notebookutils.fs.cp(
        zip_path,
        zip_locale_fs
    )

    if not copia_eseguita:
        raise RuntimeError(
            f"Impossibile copiare lo ZIP in locale: {zip_path}"
        )

    print(f"ZIP copiato in locale: {zip_locale_python}")

    # Apre lo ZIP locale
    with zipfile.ZipFile(zip_locale_python, mode="r") as archivio_zip:

        # Controlla preliminarmente che lo ZIP non sia danneggiato
        file_danneggiato = archivio_zip.testzip()

        if file_danneggiato is not None:
            raise zipfile.BadZipFile(
                f"File danneggiato nello ZIP: {file_danneggiato}"
            )

        # Seleziona gli XML, comprese eventuali sottocartelle nello ZIP
        file_xml = [
            zip_info
            for zip_info in archivio_zip.infolist()
            if (
                not zip_info.is_dir()
                and zip_info.filename.lower().endswith(".xml")
            )
        ]

        print(f"XML trovati nello ZIP: {len(file_xml)}")

        for zip_info in file_xml:

            nome_file = zip_info.filename

            try:
                # Legge l'XML come byte direttamente dallo ZIP.
                # ET.fromstring gestisce la codifica dichiarata nell'XML.
                xml_bytes = archivio_zip.read(zip_info)

                # Converte il contenuto nell'albero XML
                root = ET.fromstring(xml_bytes)

                # Considera soltanto documenti FPR12
                versione = root.attrib.get("versione", "").upper()

                if versione != "FPR12":
                    del xml_bytes
                    continue

                # Elimina completamente tutti i blocchi <Allegati>.
                # Gestisce anche XML contenenti più FatturaElettronicaBody.
                for body in root.findall(
                    ".//{*}FatturaElettronicaBody"
                ):
                    for allegati in body.findall("{*}Allegati"):
                        body.remove(allegati)

                # Libera i byte originali, che contenevano ancora gli allegati
                del xml_bytes

                # Conserva in memoria l'albero XML alleggerito
                xml_in_memoria.append({
                    "File": nome_file,
                    "Path": f"{zip_path}!/{nome_file}",
                    "Root": root
                })

                # Legge la denominazione del fornitore
                elemento = root.find(
                    ".//{*}CedentePrestatore/"
                    "{*}DatiAnagrafici/"
                    "{*}Anagrafica/"
                    "{*}Denominazione"
                )

                if elemento is not None and elemento.text:
                    denominazione = elemento.text.strip()

                else:
                    # Se il fornitore è una persona fisica,
                    # utilizza Nome e Cognome
                    nome = root.find(
                        ".//{*}CedentePrestatore/"
                        "{*}DatiAnagrafici/"
                        "{*}Anagrafica/"
                        "{*}Nome"
                    )

                    cognome = root.find(
                        ".//{*}CedentePrestatore/"
                        "{*}DatiAnagrafici/"
                        "{*}Anagrafica/"
                        "{*}Cognome"
                    )

                    denominazione = " ".join(
                        elemento_anagrafico.text.strip()
                        for elemento_anagrafico in [nome, cognome]
                        if (
                            elemento_anagrafico is not None
                            and elemento_anagrafico.text
                            and elemento_anagrafico.text.strip()
                        )
                    )

                    if not denominazione:
                        denominazione = (
                            "DENOMINAZIONE NON TROVATA"
                        )

                denominazioni.append(denominazione)

            except Exception as errore:
                # Evita di mantenere riferimenti a dati parziali
                if "xml_bytes" in locals():
                    del xml_bytes

                errori.append({
                    "File": nome_file,
                    "Errore": str(errore)
                })

except FileNotFoundError as errore:
    raise FileNotFoundError(
        f"Il file xml.zip non è stato trovato in: {zip_path}"
    ) from errore

except zipfile.BadZipFile as errore:
    raise RuntimeError(
        f"Il file {zip_path} non è uno ZIP valido o è danneggiato."
    ) from errore

finally:
    # Elimina la copia temporanea locale dello ZIP
    if notebookutils.fs.exists(zip_locale_fs):
        notebookutils.fs.rm(zip_locale_fs)


# Crea il DataFrame con il conteggio degli XML per fornitore
df = (
    pd.Series(
        denominazioni,
        name="Denominazione",
        dtype="string"
    )
    .value_counts()
    .rename_axis("Denominazione")
    .reset_index(name="Count_XML")
    .sort_values("Count_XML", ascending=False)
    .reset_index(drop=True)
)

display(df)


# Eventuali XML non elaborati
df_errori = pd.DataFrame(
    errori,
    columns=["File", "Errore"]
)

if not df_errori.empty:
    print(f"XML non elaborati: {len(df_errori)}")
    display(df_errori)


# Riepilogo
print(f"XML presenti nello ZIP: {len(file_xml)}")
print(
    "XML FPR12 conservati in memoria: "
    f"{len(xml_in_memoria)}"
)

# %% [3]


# ==============================================================
# FUNZIONE DI SUPPORTO
# ==============================================================

def testo(elemento, default=None):
    """
    Restituisce il testo pulito di un elemento XML.
    """

    if elemento is not None and elemento.text:
        return elemento.text.strip()

    return default


# Una voce per ogni DettaglioLinee
righe_xml_fatture = []

# Una voce per ogni CodiceArticolo
codici_xml_articolo = []

# Chiave tecnica progressiva della riga XML
chiave_riga_xml = 0

# Righe con una data DDT valorizzata e righe con date discordanti
righe_con_data_ddt = 0
righe_con_date_ddt_discordanti = 0


# ==============================================================
# LETTURA DEGLI XML CONSERVATI IN MEMORIA
# ==============================================================

for documento_xml in xml_in_memoria:

    nome_file_xml = documento_xml["File"]
    root_xml = documento_xml["Root"]

    # ======================================================
    # FORNITORE (CedentePrestatore)
    # ======================================================
    # La denominazione sta nell'intestazione del file, non nel
    # body: vale quindi per tutte le linee, anche quando il file
    # contiene più FatturaElettronicaBody.

    denominazione_fornitore = testo(
        root_xml.find(
            ".//{*}CedentePrestatore/"
            "{*}DatiAnagrafici/"
            "{*}Anagrafica/"
            "{*}Denominazione"
        )
    )

    if not denominazione_fornitore:

        # Persona fisica: Nome + Cognome
        denominazione_fornitore = " ".join(
            valore.strip()
            for valore in (
                testo(
                    root_xml.find(
                        ".//{*}CedentePrestatore/"
                        "{*}DatiAnagrafici/"
                        "{*}Anagrafica/"
                        "{*}Nome"
                    )
                ),
                testo(
                    root_xml.find(
                        ".//{*}CedentePrestatore/"
                        "{*}DatiAnagrafici/"
                        "{*}Anagrafica/"
                        "{*}Cognome"
                    )
                ),
            )
            if valore
        ) or None

    # Uno stesso XML può contenere più FatturaElettronicaBody
    bodies_xml = root_xml.findall(
        ".//{*}FatturaElettronicaBody"
    )

    for indice_body_xml, body_xml in enumerate(
        bodies_xml,
        start=1
    ):

        # ======================================================
        # DATI GENERALI DELLA FATTURA
        # ======================================================

        dati_documento_xml = body_xml.find(
            "./{*}DatiGenerali/"
            "{*}DatiGeneraliDocumento"
        )

        if dati_documento_xml is not None:

            tipo_documento = testo(
                dati_documento_xml.find(
                    "./{*}TipoDocumento"
                )
            )

            divisa = testo(
                dati_documento_xml.find(
                    "./{*}Divisa"
                )
            )

            data_fattura = testo(
                dati_documento_xml.find(
                    "./{*}Data"
                )
            )

            numero_fattura = testo(
                dati_documento_xml.find(
                    "./{*}Numero"
                )
            )

        else:
            tipo_documento = None
            divisa = None
            data_fattura = None
            numero_fattura = None

        # ======================================================
        # LETTURA DELLE RIGHE DELLA FATTURA
        # ======================================================

        dettagli_linee_xml = body_xml.findall(
            "./{*}DatiBeniServizi/"
            "{*}DettaglioLinee"
        )

        # ======================================================
        # COSTRUZIONE DELLA MAPPA TRA LINEE E DDT
        # ======================================================

        blocchi_ddt_xml = body_xml.findall(
            "./{*}DatiGenerali/"
            "{*}DatiDDT"
        )

        # DDT con riferimento esplicito alla linea
        ddt_espliciti_per_linea = {}

        # DDT senza alcun RiferimentoNumeroLinea
        ddt_impliciti = []

        # Date dichiarate per ciascun NumeroDDT. Lo stesso numero
        # può comparire in più blocchi DatiDDT: le date vengono
        # raccolte tutte e la linea ne riceve una solo se univoca.
        date_ddt_per_numero = {}

        # Dati per stabilire se il match implicito è ammesso
        numeri_ddt_distinti = set()
        numero_riferimenti_riga = 0

        for blocco_ddt_xml in blocchi_ddt_xml:

            numero_ddt = testo(
                blocco_ddt_xml.find(
                    "./{*}NumeroDDT"
                )
            )

            # DataDDT è facoltativo nel tracciato (2.1.8.2, xs:date
            # in formato YYYY-MM-DD): se il fornitore non lo
            # valorizza la data resta sconosciuta.
            data_ddt = testo(
                blocco_ddt_xml.find(
                    "./{*}DataDDT"
                )
            )

            if numero_ddt and data_ddt:
                date_ddt_per_numero.setdefault(numero_ddt, set()).add(
                    data_ddt
                )

            riferimenti_linea = [
                riferimento.text.strip()
                for riferimento
                in blocco_ddt_xml.findall(
                    "./{*}RiferimentoNumeroLinea"
                )
                if riferimento.text
            ]

            # Accumula le informazioni per la validazione
            # del match implicito
            if numero_ddt:
                numeri_ddt_distinti.add(numero_ddt)

            numero_riferimenti_riga += len(riferimenti_linea)

            if riferimenti_linea:

                for numero_linea_riferito in riferimenti_linea:

                    ddt_espliciti_per_linea.setdefault(
                        numero_linea_riferito,
                        []
                    )

                    if numero_ddt:
                        ddt_espliciti_per_linea[
                            numero_linea_riferito
                        ].append(numero_ddt)

            elif numero_ddt:

                # In assenza di riferimenti, il DDT
                # viene considerato valido per tutte le linee
                ddt_impliciti.append(numero_ddt)

        # Elimina eventuali duplicati mantenendo l'ordine
        ddt_impliciti = list(
            dict.fromkeys(ddt_impliciti)
        )

        # Il match implicito è ammesso soltanto se esiste
        # UN SOLO NumeroDDT distinto e NESSUN RiferimentoNumeroLinea.
        #
        # La condizione è valutata sull'intero documento
        # (IdFile + NumeroFattura) e vale solo se sono vere
        # entrambe:
        #
        #     NumeroDDTDistinti     = 1
        #     NumeroRiferimentiRiga = 0
        #
        # In questo modo il DDT implicito non viene più applicato
        # alle linee non citate quando esistono altri blocchi
        # DatiDDT con RiferimentoNumeroLinea valorizzato.
        if not (
            len(numeri_ddt_distinti) == 1
            and numero_riferimenti_riga == 0
        ):
            ddt_impliciti = []

        # ======================================================
        # CREAZIONE DELLE RIGHE DEL DATAFRAME PRINCIPALE
        # ======================================================

        for dettaglio_linea_xml in dettagli_linee_xml:

            # Genera una chiave tecnica univoca
            chiave_riga_xml += 1

            numero_linea = testo(
                dettaglio_linea_xml.find(
                    "./{*}NumeroLinea"
                )
            )

            # --------------------------------------------------
            # ABBINAMENTO DEL DDT ALLA LINEA
            # --------------------------------------------------

            ddt_espliciti_linea = (
                ddt_espliciti_per_linea.get(
                    numero_linea,
                    []
                )
            )

            ddt_espliciti_linea = list(
                dict.fromkeys(ddt_espliciti_linea)
            )

            if ddt_espliciti_linea:

                numeri_ddt_linea = ddt_espliciti_linea
                tipo_abbinamento_ddt = "ESPLICITO"

            elif ddt_impliciti:

                numeri_ddt_linea = ddt_impliciti
                tipo_abbinamento_ddt = "IMPLICITO"

            elif blocchi_ddt_xml:

                numeri_ddt_linea = []
                tipo_abbinamento_ddt = "NON_ABBINATO"

            else:

                numeri_ddt_linea = []
                tipo_abbinamento_ddt = "NESSUN_DDT"

            numero_ddt_linea = (
                " | ".join(numeri_ddt_linea)
                if numeri_ddt_linea
                else None
            )

            # --------------------------------------------------
            # DATA DEL DDT
            # --------------------------------------------------
            # La linea riceve la data solo se le date dei DDT che
            # la riguardano coincidono. Con più date diverse il
            # dato è ambiguo e resta vuoto: l'abbinamento non deve
            # poggiare su una data scelta a caso.

            date_linea = {
                data
                for numero_ddt_corrente in numeri_ddt_linea
                for data in date_ddt_per_numero.get(
                    numero_ddt_corrente,
                    ()
                )
            }

            if len(date_linea) > 1:
                righe_con_date_ddt_discordanti += 1

            data_ddt_linea = (
                next(iter(date_linea))
                if len(date_linea) == 1
                else None
            )

            if data_ddt_linea is not None:
                righe_con_data_ddt += 1

            # --------------------------------------------------
            # RIGA DEL DATAFRAME PRINCIPALE
            # --------------------------------------------------

            righe_xml_fatture.append({

                # Chiave tecnica
                "ChiaveRigaXML": chiave_riga_xml,

                # Provenienza XML
                "FileXML": nome_file_xml,
                "IndiceBodyXML": indice_body_xml,

                # Fornitore
                "Denominazione": denominazione_fornitore,

                # Dati generali della fattura
                "TipoDocumento": tipo_documento,
                "Divisa": divisa,
                "DataFattura": data_fattura,
                "NumeroFattura": numero_fattura,

                # Dati della singola linea
                "NumeroLinea": numero_linea,

                "TipoCessionePrestazione": testo(
                    dettaglio_linea_xml.find(
                        "./{*}TipoCessionePrestazione"
                    )
                ),

                "Descrizione": testo(
                    dettaglio_linea_xml.find(
                        "./{*}Descrizione"
                    )
                ),

                "Quantita": testo(
                    dettaglio_linea_xml.find(
                        "./{*}Quantita"
                    )
                ),

                "UnitaMisura": testo(
                    dettaglio_linea_xml.find(
                        "./{*}UnitaMisura"
                    )
                ),

                "PrezzoUnitario": testo(
                    dettaglio_linea_xml.find(
                        "./{*}PrezzoUnitario"
                    )
                ),

                "PrezzoTotale": testo(
                    dettaglio_linea_xml.find(
                        "./{*}PrezzoTotale"
                    )
                ),

                "RiferimentoAmministrazione": testo(
                    dettaglio_linea_xml.find(
                        "./{*}RiferimentoAmministrazione"
                    )
                ),

                # DDT della linea
                "NumeroDDT": numero_ddt_linea,
                "DataDDT": data_ddt_linea,
                "TipoAbbinamentoDDT":
                    tipo_abbinamento_ddt
            })

            # ==================================================
            # CODICI ARTICOLO DELLA LINEA
            # ==================================================

            blocchi_codice_articolo = (
                dettaglio_linea_xml.findall(
                    "./{*}CodiceArticolo"
                )
            )

            for progressivo_codice, codice_articolo_xml in enumerate(
                blocchi_codice_articolo,
                start=1
            ):

                codici_xml_articolo.append({

                    # Collegamento con df_xml_fatture
                    "ChiaveRigaXML": chiave_riga_xml,

                    # Ordine del codice dentro la linea XML
                    "ProgressivoCodiceXML":
                        progressivo_codice,

                    # Valori originali, non normalizzati
                    "CodiceTipoXML": testo(
                        codice_articolo_xml.find(
                            "./{*}CodiceTipo"
                        )
                    ),

                    "CodiceValoreXML": testo(
                        codice_articolo_xml.find(
                            "./{*}CodiceValore"
                        )
                    )
                })

            # ==================================================
            # RIFERIMENTO AMMINISTRAZIONE COME CODICE ARTICOLO
            # ==================================================

            riferimento_amministrazione = testo(
                dettaglio_linea_xml.find(
                    "./{*}RiferimentoAmministrazione"
                )
            )

            if riferimento_amministrazione:

                codici_xml_articolo.append({

                    # Collegamento con df_xml_fatture
                    "ChiaveRigaXML": chiave_riga_xml,

                    # Posizione successiva agli eventuali codici articolo
                    "ProgressivoCodiceXML":
                        len(blocchi_codice_articolo) + 1,

                    # Tipo attribuito al riferimento amministrativo
                    "CodiceTipoXML":
                        "RiferimentoAmministrazione",

                    # Valore originale del nodo XML
                    "CodiceValoreXML":
                        riferimento_amministrazione
                })

            # NOTA - la descrizione diventa un codice articolo
            # nella cella [12], non qui: decidere se serve dipende
            # dal dizionario, che a questo punto non è ancora
            # stato letto.


# ==============================================================
# CREAZIONE DEL DATAFRAME PRINCIPALE
# ==============================================================

colonne_df_xml_fatture = [
    "ChiaveRigaXML",
    "FileXML",
    "IndiceBodyXML",
    "Denominazione",
    "TipoDocumento",
    "Divisa",
    "DataFattura",
    "NumeroFattura",
    "NumeroLinea",
    "TipoCessionePrestazione",
    "Descrizione",
    "Quantita",
    "UnitaMisura",
    "PrezzoUnitario",
    "PrezzoTotale",
    "RiferimentoAmministrazione",
    "NumeroDDT",
    "DataDDT",
    "TipoAbbinamentoDDT"
]

df_xml_fatture = pd.DataFrame(
    righe_xml_fatture,
    columns=colonne_df_xml_fatture
)


# ==============================================================
# CREAZIONE DEL DATAFRAME LONG DEI CODICI ARTICOLO
# ==============================================================

colonne_df_xml_codici_articolo = [
    "ChiaveRigaXML",
    "ProgressivoCodiceXML",
    "CodiceTipoXML",
    "CodiceValoreXML"
]

df_xml_codici_articolo = pd.DataFrame(
    codici_xml_articolo,
    columns=colonne_df_xml_codici_articolo
)


# ==============================================================
# CONVERSIONE DELLA DATA
# ==============================================================

df_xml_fatture["DataFattura"] = pd.to_datetime(
    df_xml_fatture["DataFattura"],
    errors="coerce"
)

# DataDDT arriva dall'XML come testo ISO (YYYY-MM-DD). Le righe
# senza data o con date discordanti restano NaT.
df_xml_fatture["DataDDT"] = pd.to_datetime(
    df_xml_fatture["DataDDT"],
    errors="coerce"
)


# ==============================================================
# CONVERSIONE DEI CAMPI NUMERICI
# ==============================================================

colonne_numeriche_xml_fatture = [
    "ChiaveRigaXML",
    "IndiceBodyXML",
    "NumeroLinea",
    "Quantita",
    "PrezzoUnitario",
    "PrezzoTotale"
]

for colonna_xml in colonne_numeriche_xml_fatture:

    df_xml_fatture[colonna_xml] = pd.to_numeric(
        df_xml_fatture[colonna_xml],
        errors="coerce"
    )


colonne_numeriche_xml_codici = [
    "ChiaveRigaXML",
    "ProgressivoCodiceXML"
]

for colonna_xml in colonne_numeriche_xml_codici:

    df_xml_codici_articolo[colonna_xml] = pd.to_numeric(
        df_xml_codici_articolo[colonna_xml],
        errors="coerce"
    )


# ==============================================================
# ORDINAMENTO
# ==============================================================

df_xml_fatture = (
    df_xml_fatture
    .sort_values(
        by=[
            "FileXML",
            "IndiceBodyXML",
            "NumeroLinea"
        ],
        na_position="last"
    )
    .reset_index(drop=True)
)


df_xml_codici_articolo = (
    df_xml_codici_articolo
    .sort_values(
        by=[
            "ChiaveRigaXML",
            "ProgressivoCodiceXML"
        ],
        na_position="last"
    )
    .reset_index(drop=True)
)


# ==============================================================
# VISUALIZZAZIONE
# ==============================================================

print("Righe delle fatture XML:")
display(df_xml_fatture)

print("Codici articolo XML:")
display(df_xml_codici_articolo)


# ==============================================================
# RIEPILOGO
# ==============================================================

print(
    f"File XML presenti in memoria: "
    f"{len(xml_in_memoria)}"
)

print(
    f"Righe XML estratte: "
    f"{len(df_xml_fatture)}"
)

print(
    f"Codici articolo XML estratti: "
    f"{len(df_xml_codici_articolo)}"
)

print(
    "Righe con data DDT valorizzata: "
    f"{righe_con_data_ddt:,} su {len(df_xml_fatture):,}"
)

print(
    "Righe con date DDT discordanti (data lasciata vuota): "
    f"{righe_con_date_ddt_discordanti:,}"
)

print("\nDistribuzione abbinamenti DDT:")

display(
    df_xml_fatture[
        "TipoAbbinamentoDDT"
    ]
    .value_counts(dropna=False)
    .rename_axis("TipoAbbinamentoDDT")
    .reset_index(name="NumeroRighe")
)

# %% [4]

# ==============================================================
# NORMALIZZAZIONE
# ==============================================================

# ==============================================================
# COSTANTI
# ==============================================================

# Separatori rimossi dalla regola RIMUOVI_SEPARATORI
SEPARATORI = " -/._:"

# Cifre ammesse (evita isdigit(), che accetta anche cifre unicode)
CIFRE = "0123456789"

# Regola applicata quando il dizionario non ne indica una.
# L'elenco delle regole disponibili è più sotto, nel REGISTRO.
REGOLA_DEFAULT = "RIMUOVI_SEPARATORI"


# ==============================================================
# FUNZIONI DI SUPPORTO INTERNE
# ==============================================================

def in_testo(valore):
    """
    Converte un valore in testo.

    Gestisce i valori nulli di pandas (None, NaN, NaT, NA) e
    converte i numeri interi letti come float senza aggiungere
    il ".0" finale, che falserebbe le chiavi del dizionario.
    """

    if valore is None or pd.isna(valore):
        return None

    if isinstance(valore, float) and valore.is_integer():
        return str(int(valore))

    return str(valore)


def valore_pulito(valore):
    """
    Livello 1 della normalizzazione, sempre applicato:

        Text.From + Text.Clean + Text.Trim
        + rimozione dello spazio non separabile (char 160)
        + maiuscolo
    """

    testo = in_testo(valore)

    if testo is None:
        return None

    # Text.Clean rimuove i caratteri non stampabili (categoria
    # unicode Cc, che comprende i codici 0-31 e 127-159).
    testo = "".join(
        carattere
        for carattere in testo
        if unicodedata.category(carattere) != "Cc"
    )

    testo = testo.strip().replace("\u00a0", "")

    if testo == "":
        return None

    return testo.upper()


def regola_normalizzata(regola):
    """
    Rende la regola maiuscola e priva di spazi.
    Una regola vuota o assente restituisce None:
    sarà applicata REGOLA_DEFAULT.
    """

    testo = in_testo(regola)

    if testo is None:
        return None

    testo = testo.strip().upper()

    return testo if testo != "" else None


def riduci_zeri(testo):
    """
    Rimuove gli zeri iniziali.
    Se il risultato è vuoto restituisce "0".
    """

    if testo is None:
        return None

    senza_zeri = testo.lstrip("0")

    return "0" if senza_zeri == "" else senza_zeri


def solo_cifre(testo):
    """
    Mantiene soltanto le cifre.
    """

    return "".join(carattere for carattere in testo if carattere in CIFRE)


def finalizza(testo):
    """
    Normalizzazione finale, condivisa da più regole:

        rimuove i separatori " - / . _ :",
        rimuove gli zeri iniziali se il valore è interamente numerico.
    """

    if testo is None:
        return None

    senza_separatori = "".join(
        carattere
        for carattere in testo
        if carattere not in SEPARATORI
    )

    if senza_separatori == "":
        return None

    if solo_cifre(senza_separatori) == senza_separatori:
        return riduci_zeri(senza_separatori)

    return senza_separatori


def prima_stringa(testo):
    """
    Parte fino al primo carattere non alfanumerico.
    Il confronto è ordinale: le lettere accentate sono quindi
    considerate separatori.
    """

    for indice, carattere in enumerate(testo):

        alfanumerico = (
            carattere in CIFRE
            or "A" <= carattere <= "Z"
        )

        if not alfanumerico:
            return testo[:indice]

    return testo


def dividi(testo, separatori):
    """
    Equivalente di Text.SplitAny: divide il testo in
    corrispondenza di uno qualsiasi dei separatori indicati.
    """

    parti = [testo]

    for separatore in separatori:

        parti_estese = []

        for parte in parti:
            parti_estese.extend(parte.split(separatore))

        parti = parti_estese

    return parti


# ==============================================================
# LE REGOLE
# ==============================================================
# Ogni regola è una funzione che riceve il valore dopo la pulizia
# di livello 1 e restituisce la chiave di confronto.
#
# Per aggiungere o modificare una regola si tocca SOLO il
# registro qui sotto: la normalizzazione e la diagnostica lo
# leggono da soli. Prima servivano tre modifiche in due punti
# diversi (l'elenco delle regole e il ramo della funzione), con
# il rischio concreto di dimenticarne uno.
#
# REGOLE[<nome>] = (funzione, descrizione)
# La descrizione viene stampata nella sezione [4]: la
# documentazione non può divergere dal codice.

def regola_nessuna(testo):
    """Nessuna trasformazione: resta la sola pulizia di livello 1."""

    return testo


def regola_rimuovi_separatori(testo):
    """
    Rimuove i separatori e gli zeri iniziali se il valore è
    interamente numerico.
    """

    return finalizza(testo)


def regola_solo_numeri(testo):
    """Tiene solo le cifre, poi toglie gli zeri iniziali."""

    cifre = solo_cifre(testo)

    if cifre == "":
        return None

    return riduci_zeri(cifre)


def regola_estrai_prima(testo):
    """Parte fino al primo carattere non alfanumerico."""

    estratto = prima_stringa(testo)

    return finalizza(None if estratto == "" else estratto)


def regola_estrai_ultima(testo):
    """Ultima parte dopo spazio, trattino o slash."""

    parti = [
        parte
        for parte in dividi(testo, " -/")
        if parte != ""
    ]

    return finalizza(parti[-1] if parti else None)


REGOLE = {
    "RIMUOVI_SEPARATORI": (
        regola_rimuovi_separatori,
        "rimuove i separatori e gli zeri iniziali se il valore è numerico",
    ),
    "RIMUOVI_ZERI": (
        regola_rimuovi_separatori,
        "come RIMUOVI_SEPARATORI",
    ),
    "NESSUNA_REGOLA": (
        regola_nessuna,
        "solo pulizia di base: trim, maiuscolo, caratteri non stampabili",
    ),
    "TESTO_MAIUSCOLO": (
        regola_nessuna,
        "come NESSUNA_REGOLA",
    ),
    "SOLO_NUMERI": (
        regola_solo_numeri,
        "tiene solo le cifre, poi toglie gli zeri iniziali",
    ),
    "ESTRAI_PRIMA_STRINGA": (
        regola_estrai_prima,
        "parte fino al primo carattere non alfanumerico",
    ),
    "ESTRAI_ULTIMA_STRINGA": (
        regola_estrai_ultima,
        "ultima parte separata da spazio, trattino o slash",
    ),
}


def regola_disponibile(nome):
    """
    Vero se il nome corrisponde a una regola del registro.
    """

    return nome in REGOLE


# ==============================================================
# NORMALIZZAZIONE
# ==============================================================

def normalizza(valore, regola=None):
    """
    Normalizza un DDT o un codice articolo.

    Le regole disponibili e il loro effetto stanno nel REGISTRO
    più sopra: qui si cerca la funzione per nome e la si applica,
    senza elenchi duplicati da tenere allineati.

    Regola non riconosciuta: NON blocca l'esecuzione,
    applica REGOLA_DEFAULT.
    """

    # ------------------------------------------------------
    # LIVELLO 1 - pulizia tecnica minima, sempre applicata
    # ------------------------------------------------------

    base = valore_pulito(valore)

    if base is None:
        return None

    # ------------------------------------------------------
    # LIVELLO 2 - applicazione della regola
    # ------------------------------------------------------

    regola_applicata = regola_normalizzata(regola) or REGOLA_DEFAULT

    # Una regola non riconosciuta ricade sul default: il
    # comportamento è identico a quello di una cella vuota.
    funzione, _ = REGOLE.get(regola_applicata, REGOLE[REGOLA_DEFAULT])

    return funzione(base)


# ==============================================================
# DIAGNOSTICA DELLE REGOLE
# ==============================================================

def stato_regola(regola):
    """
    Serve solo alla segnalazione: una regola scritta male non
    blocca l'esecuzione, quindi senza questo controllo l'errore
    diventerebbe silenzioso.

    Restituisce:

        "OK"                          regola valida
        "NESSUNA REGOLA - DEFAULT"    cella vuota: si applica il default
        "NON RICONOSCIUTA: <valore>"  valore non ammesso: si applica il default
    """

    regola_applicata = regola_normalizzata(regola)

    if regola_applicata is None:
        return "NESSUNA REGOLA - DEFAULT"

    if regola_applicata in REGOLE:
        return "OK"

    return f"NON RICONOSCIUTA: {regola_applicata}"


# ==============================================================
# TABELLA DELLE REGOLE
# ==============================================================
# Stampa il registro: è la sola fonte di verità sulle regole
# disponibili, quindi non serve tenerne una copia a mano nei
# documenti.

print(f"Regola applicata quando il dizionario non ne indica una: {REGOLA_DEFAULT}")
print()

display(
    pd.DataFrame(
        [
            {"Regola": nome, "Effetto": descrizione}
            for nome, (_, descrizione) in REGOLE.items()
        ]
    )
)

# %% [5]
# ==============================================================
# DIZIONARI
# ==============================================================
# Struttura a fogli (file "Dizionario codici.xlsx"):
#
#   RegoleCodiciXML      CFOR + CodiceTipoXML  → CampoCanonico, regola
#   RegoleCodiciCarico   CFOR + CampoCanonico  → regola
#   RegoleDDT            CFOR                  → regola XML, regola carico
#
# I due lati hanno granularità diverse, da qui i fogli separati:
# sul lato XML uno stesso CampoCanonico può essere raggiunto da più
# CodiceTipoXML (es. biG), sul lato carico la coppia
# (CFOR, CampoCanonico) deve essere unica.
#
# Una riga priva di CFOR viene semplicemente non considerata.
# ==============================================================

# Nomi delle colonne attesi nei fogli
COLONNE_CODICI_XML = {
    "cfor": "CFOR",
    "denominazione": "Denominazione",
    "codice_tipo": "CodiceTipoXML",
    "campo_canonico": "CampoCanonico",
    "regola": "RegolaNormalizzazione",
}

COLONNE_CODICI_CARICO = {
    "cfor": "CFOR",
    "denominazione": "Denominazione",
    "campo_canonico": "CampoCanonico",
    "regola": "RegolaNormalizzazione",
}

COLONNE_DDT = {
    "cfor": "CFOR",
    "denominazione": "Denominazione",
    "regola_xml": "RegolaDDTXML",
    "regola_carico": "RegolaDDTCarico",
}


def chiave_cfor(cfor):
    """
    Chiave del fornitore: trim e maiuscolo.
    """

    testo = in_testo(cfor)

    if testo is None:
        return None

    testo = testo.strip()

    return testo.upper() if testo != "" else None


def chiave_testo(valore):
    """
    Chiave testuale case-insensitive: trim e maiuscolo.
    """

    testo = in_testo(valore)

    if testo is None:
        return None

    testo = testo.strip()

    return testo.upper() if testo != "" else None


def _leggi(df, nome_colonna):
    """
    Restituisce la colonna richiesta, oppure None se non esiste.
    """

    if df is None or nome_colonna not in df.columns:
        return None

    return df[nome_colonna]


def prepara_dizionari(
    df_codici_xml,
    df_codici_carico=None,
    df_regole_ddt=None,
    colonne_xml=None,
    colonne_carico=None,
    colonne_ddt=None,
):
    """
    Costruisce le mappe di lookup e la diagnostica.

    Restituisce un dizionario con:

        "CodiciXML"        (CFOR, CODICETIPO) → (CampoCanonico, regola)
        "CodiciCarico"     (CFOR, CAMPOCANONICO) → regola
        "DDT"              CFOR → (regola XML, regola carico)
        "Anomalie"         regole non riconosciute
        "ChiaviDuplicate"  chiavi ripetute
    """

    colonne_xml = colonne_xml or COLONNE_CODICI_XML
    colonne_carico = colonne_carico or COLONNE_CODICI_CARICO
    colonne_ddt = colonne_ddt or COLONNE_DDT

    mappa_codici_xml = {}
    mappa_codici_carico = {}
    mappa_ddt = {}

    anomalie = []
    chiavi_duplicate = []

    # ------------------------------------------------------
    # Foglio RegoleCodiciXML
    # ------------------------------------------------------

    if df_codici_xml is not None:

        col_cfor = _leggi(df_codici_xml, colonne_xml["cfor"])
        col_tipo = _leggi(df_codici_xml, colonne_xml["codice_tipo"])
        col_campo = _leggi(df_codici_xml, colonne_xml["campo_canonico"])
        col_regola = _leggi(df_codici_xml, colonne_xml["regola"])

        for indice in df_codici_xml.index:

            cfor = chiave_cfor(None if col_cfor is None else col_cfor[indice])

            # Una riga priva di CFOR non viene considerata
            if cfor is None:
                continue

            tipo = chiave_testo(None if col_tipo is None else col_tipo[indice])

            if tipo is None:
                continue

            campo = chiave_testo(None if col_campo is None else col_campo[indice])
            regola = in_testo(None if col_regola is None else col_regola[indice])

            chiave = (cfor, tipo)

            if chiave in mappa_codici_xml:
                chiavi_duplicate.append({
                    "Foglio": "RegoleCodiciXML",
                    "CFOR": cfor,
                    "Valore": tipo,
                })

            # In caso di duplicato vale la prima riga letta
            mappa_codici_xml.setdefault(chiave, (campo, regola))

            stato = stato_regola(regola)

            if stato != "OK":
                anomalie.append({
                    "Foglio": "RegoleCodiciXML",
                    "CFOR": cfor,
                    "Colonna": "RegolaNormalizzazione",
                    "RegolaScritta": regola,
                    "Stato": stato,
                })

    # ------------------------------------------------------
    # Foglio RegoleCodiciCarico
    # ------------------------------------------------------

    if df_codici_carico is not None:

        col_cfor = _leggi(df_codici_carico, colonne_carico["cfor"])
        col_campo = _leggi(df_codici_carico, colonne_carico["campo_canonico"])
        col_regola = _leggi(df_codici_carico, colonne_carico["regola"])

        for indice in df_codici_carico.index:

            cfor = chiave_cfor(None if col_cfor is None else col_cfor[indice])

            if cfor is None:
                continue

            campo = chiave_testo(None if col_campo is None else col_campo[indice])

            if campo is None:
                continue

            regola = in_testo(None if col_regola is None else col_regola[indice])

            chiave = (cfor, campo)

            if chiave in mappa_codici_carico:
                chiavi_duplicate.append({
                    "Foglio": "RegoleCodiciCarico",
                    "CFOR": cfor,
                    "Valore": campo,
                })

            mappa_codici_carico.setdefault(chiave, regola)

            stato = stato_regola(regola)

            if stato != "OK":
                anomalie.append({
                    "Foglio": "RegoleCodiciCarico",
                    "CFOR": cfor,
                    "Colonna": "RegolaNormalizzazione",
                    "RegolaScritta": regola,
                    "Stato": stato,
                })

    # ------------------------------------------------------
    # Foglio RegoleDDT
    # ------------------------------------------------------

    if df_regole_ddt is not None:

        col_cfor = _leggi(df_regole_ddt, colonne_ddt["cfor"])
        col_xml = _leggi(df_regole_ddt, colonne_ddt["regola_xml"])
        col_carico = _leggi(df_regole_ddt, colonne_ddt["regola_carico"])

        for indice in df_regole_ddt.index:

            cfor = chiave_cfor(None if col_cfor is None else col_cfor[indice])

            if cfor is None:
                continue

            regola_xml = in_testo(None if col_xml is None else col_xml[indice])
            regola_carico = in_testo(
                None if col_carico is None else col_carico[indice]
            )

            if cfor in mappa_ddt:
                chiavi_duplicate.append({
                    "Foglio": "RegoleDDT",
                    "CFOR": cfor,
                    "Valore": cfor,
                })

            mappa_ddt.setdefault(cfor, (regola_xml, regola_carico))

            for nome_colonna, regola in (
                ("RegolaDDTXML", regola_xml),
                ("RegolaDDTCarico", regola_carico),
            ):

                stato = stato_regola(regola)

                if stato != "OK":
                    anomalie.append({
                        "Foglio": "RegoleDDT",
                        "CFOR": cfor,
                        "Colonna": nome_colonna,
                        "RegolaScritta": regola,
                        "Stato": stato,
                    })

    return {
        "CodiciXML": mappa_codici_xml,
        "CodiciCarico": mappa_codici_carico,
        "DDT": mappa_ddt,
        "Anomalie": pd.DataFrame(
            anomalie,
            columns=[
                "Foglio",
                "CFOR",
                "Colonna",
                "RegolaScritta",
                "Stato",
            ],
        ),
        "ChiaviDuplicate": pd.DataFrame(
            chiavi_duplicate,
            columns=["Foglio", "CFOR", "Valore"],
        ),
    }


# ==============================================================
# RISOLUZIONE DELLE REGOLE
# ==============================================================

def regola_codice_xml(dizionari, cfor, codice_tipo):
    """
    Restituisce (CampoCanonico, Regola) per il lato XML.

    Se la coppia non è mappata restituisce (None, None):
    verrà applicata la regola di default.
    """

    chiave = (chiave_cfor(cfor), chiave_testo(codice_tipo))

    return dizionari["CodiciXML"].get(chiave, (None, None))


def regola_codice_carico(dizionari, cfor, campo_canonico):
    """
    Restituisce la regola per il lato carichi, oppure None.
    """

    chiave = (chiave_cfor(cfor), chiave_testo(campo_canonico))

    return dizionari["CodiciCarico"].get(chiave)


def regola_ddt_xml(dizionari, cfor):
    """
    Regola DDT del lato XML.
    In assenza di configurazione restituisce None:
    verrà applicata la regola di default.
    """

    return dizionari["DDT"].get(chiave_cfor(cfor), (None, None))[0]


def regola_ddt_carico(dizionari, cfor):
    """
    Regola DDT del lato carichi.
    """

    return dizionari["DDT"].get(chiave_cfor(cfor), (None, None))[1]


# ==============================================================
# FUNZIONI DI COMODO
# ==============================================================
# Uniscono risoluzione della regola e normalizzazione.
# Sono il punto di ingresso che userà il flusso del notebook.
#
# Finché il CFOR non è disponibile si può passare cfor=None:
# la regola non viene trovata e si applica il default.

def normalizza_ddt_xml(dizionari, cfor, valore):
    """
    Normalizza un NumeroDDT proveniente dall'XML.
    """

    return normalizza(valore, regola_ddt_xml(dizionari, cfor))


def normalizza_ddt_carico(dizionari, cfor, valore):
    """
    Normalizza un NDOGIAB proveniente dai carichi.
    """

    return normalizza(valore, regola_ddt_carico(dizionari, cfor))


def normalizza_codice_xml(dizionari, cfor, codice_tipo, valore):
    """
    Normalizza un CodiceValore proveniente dall'XML.

    Restituisce (CampoCanonico, valore normalizzato).

    Se il CodiceTipo non è mappato CampoCanonico è None, ma il
    valore viene comunque normalizzato con la regola di default:
    è il comportamento richiesto finché il CFOR non è agganciato.
    """

    campo_canonico, regola = regola_codice_xml(dizionari, cfor, codice_tipo)

    return (campo_canonico, normalizza(valore, regola))


def normalizza_codice_carico(dizionari, cfor, campo_canonico, valore):
    """
    Normalizza un codice proveniente dai carichi.
    """

    return normalizza(
        valore,
        regola_codice_carico(dizionari, cfor, campo_canonico),
    )


# %% [6]

# ==============================================================
# INIZIO BLOCCO DIAGNOSTICA - RIMUOVIBILE
# ==============================================================
# Cella da incollare nel notebook DOPO le due celle del modulo
# di normalizzazione.
#
# Serve soltanto a verificare il contenuto del dizionario.
#
# NON PRODUCE DATI UTILIZZATI A VALLE.
# Per rimuoverla: cancellare tutto il compreso fra
# "INIZIO BLOCCO DIAGNOSTICA" e "FINE BLOCCO DIAGNOSTICA".
# Il flusso continua a funzionare senza altre modifiche.
# ==============================================================

# --------------------------------------------------------------
# Copia e lettura
# --------------------------------------------------------------

if notebookutils.fs.exists(dizionari_fs):
    notebookutils.fs.rm(dizionari_fs)

copia_dizionari = notebookutils.fs.cp(percorso_dizionari, dizionari_fs)

if not copia_dizionari:
    raise RuntimeError(
        f"Impossibile copiare il dizionario in locale: {percorso_dizionari}"
    )

try:

    # sheet_name=None legge tutti i fogli in un'unica passata
    fogli_excel = pd.read_excel(
        dizionari_locale,
        sheet_name=None,
        dtype=str,
    )

finally:
    if notebookutils.fs.exists(dizionari_fs):
        notebookutils.fs.rm(dizionari_fs)

print(f"Fogli presenti nel file: {list(fogli_excel.keys())}")


def prendi_foglio(nome_foglio):
    """
    Restituisce il foglio richiesto, oppure None se non esiste.
    """

    if nome_foglio in fogli_excel:
        return fogli_excel[nome_foglio]

    print(f"Foglio non presente e quindi non considerato: {nome_foglio}")
    return None


df_diz_codici_xml = prendi_foglio(NOME_FOGLIO_CODICI_XML)
df_diz_codici_carico = prendi_foglio(NOME_FOGLIO_CODICI_CARICO)
df_diz_regole_ddt = prendi_foglio(NOME_FOGLIO_DDT)

# --------------------------------------------------------------
# Preparazione dei dizionari
# --------------------------------------------------------------

dizionari = prepara_dizionari(
    df_codici_xml=df_diz_codici_xml,
    df_codici_carico=df_diz_codici_carico,
    df_regole_ddt=df_diz_regole_ddt,
)

# --------------------------------------------------------------
# Stampa
# --------------------------------------------------------------

print("Righe lette / righe con CFOR per foglio:")

for nome_foglio, df_foglio in (
    (NOME_FOGLIO_CODICI_XML, df_diz_codici_xml),
    (NOME_FOGLIO_CODICI_CARICO, df_diz_codici_carico),
    (NOME_FOGLIO_DDT, df_diz_regole_ddt),
):

    if df_foglio is None:
        print(f"  {nome_foglio}: foglio assente")

    elif "CFOR" not in df_foglio.columns:
        print(f"  {nome_foglio}: colonna CFOR assente")

    else:
        print(
            f"  {nome_foglio}: "
            f"{len(df_foglio)} righe lette, "
            f"{int(df_foglio['CFOR'].notna().sum())} con CFOR"
        )

print()
print(
    "Righe attive nel dizionario codici XML: "
    f"{len(dizionari['CodiciXML'])}"
)
print(
    "Righe attive nel dizionario codici carico: "
    f"{len(dizionari['CodiciCarico'])}"
)
print(
    f"Righe attive nel dizionario DDT: {len(dizionari['DDT'])}"
)


if not dizionari["Anomalie"].empty:
    print("\nRegole non riconosciute:")
    display(dizionari["Anomalie"])
else:
    print("\nNessuna regola non riconosciuta.")


if not dizionari["ChiaviDuplicate"].empty:
    print("\nChiavi duplicate (vale la prima riga):")
    display(dizionari["ChiaviDuplicate"])
else:
    print("Nessuna chiave duplicata.")


print("\nDizionario codici XML:")

display(
    pd.DataFrame(
        [
            {
                "CFOR": cfor,
                "CodiceTipoXML": tipo,
                "CampoCanonico": campo,
                "Regola": regola,
                "Stato": stato_regola(regola),
            }
            for (cfor, tipo), (campo, regola)
            in sorted(dizionari["CodiciXML"].items())
        ],
        columns=[
            "CFOR",
            "CodiceTipoXML",
            "CampoCanonico",
            "Regola",
            "Stato",
        ],
    )
)


if dizionari["CodiciCarico"]:

    print("Dizionario codici carico:")

    display(
        pd.DataFrame(
            [
                {
                    "CFOR": cfor,
                    "CampoCanonico": campo,
                    "Regola": regola,
                    "Stato": stato_regola(regola),
                }
                for (cfor, campo), regola
                in sorted(dizionari["CodiciCarico"].items())
            ],
            columns=["CFOR", "CampoCanonico", "Regola", "Stato"],
        )
    )


print("Dizionario DDT:")

display(
    pd.DataFrame(
        [
            {
                "CFOR": cfor,
                "RegolaDDTXML": regola_xml,
                "RegolaDDTCarico": regola_carico,
                "StatoXML": stato_regola(regola_xml),
                "StatoCarico": stato_regola(regola_carico),
            }
            for cfor, (regola_xml, regola_carico)
            in sorted(dizionari["DDT"].items())
        ],
        columns=[
            "CFOR",
            "RegolaDDTXML",
            "RegolaDDTCarico",
            "StatoXML",
            "StatoCarico",
        ],
    )
)

# ==============================================================
# FINE BLOCCO DIAGNOSTICA
# ==============================================================

# %% [7]


# ==============================================================
# CARICHI - LETTURA E TIPIZZAZIONE
# ==============================================================
# Lettura della sorgente, impostazione dei tipi e filtro sul tipo
# di documento.
# ==============================================================

# ==============================================================
# SCHEMA
# ==============================================================

STRING_COLUMNS = [
    "CAZI",
    "CFOR",
    "CEDI",
    "CDOCIAA",
    "CART",
    "XART",
    "NDOGIAA",
    "NDOGIAB",
    "CSTQ",
    "CUMF",
    "CEAS",
    "CALA",
]

# CPCO è numerico: nell'anagrafica dei punti di consegna è
# tipizzato come intero, quindi va mantenuto tale per poter
# essere usato come chiave di join.
INTEGER_COLUMNS = [
    "CPCO",
]

DECIMAL_COLUMNS = [
    "Qtà",
    "Importo_ns_fattura",
    "PF02IAL",
    "QTA ORDINE",
    "QTA ACCETT",
]

DATE_COLUMNS = [
    "TDATIAA",
    "TDATIAB",
]

EXPECTED_COLUMNS = (
    STRING_COLUMNS
    + INTEGER_COLUMNS
    + DECIMAL_COLUMNS
    + DATE_COLUMNS
)

# Colonne utilizzate come chiavi nei join successivi
KEY_COLUMNS = [
    "CFOR",
    "CART",
    "NDOGIAA",
    "NDOGIAB",
    "CEAS",
]


# ==============================================================
# LETTURA SORGENTE
# ==============================================================

def read_source() -> pd.DataFrame:

    if SOURCE_TYPE == "excel":
        return pd.read_excel(
            CARICHI_PATH,
            sheet_name=CARICHI_SHEET,
            engine="openpyxl",
            dtype=object,
        )

    if SOURCE_TYPE == "table":
        return spark.table(TABLE_NAME).toPandas()

    raise ValueError(
        f"Sorgente non supportata: {SOURCE_TYPE}"
    )


# ==============================================================
# TIPIZZAZIONE
# ==============================================================

def apply_schema(df: pd.DataFrame) -> pd.DataFrame:

    result = df.copy()

    result.columns = (
        result.columns
        .astype(str)
        .str.strip()
    )

    for column in STRING_COLUMNS:
        if column in result.columns:
            result[column] = (
                result[column]
                .astype("string")
                .str.strip()
            )

    for column in INTEGER_COLUMNS:
        if column in result.columns:
            result[column] = pd.to_numeric(
                result[column],
                errors="coerce",
            ).astype("Int64")

    for column in DECIMAL_COLUMNS:
        if column in result.columns:
            result[column] = pd.to_numeric(
                result[column],
                errors="coerce",
            ).astype("Float64")

    for column in DATE_COLUMNS:
        if column in result.columns:
            result[column] = pd.to_datetime(
                result[column],
                errors="coerce",
            )

    return result


# ==============================================================
# VALIDAZIONE
# ==============================================================

def validate_schema(df: pd.DataFrame) -> None:

    missing_columns = sorted(
        set(EXPECTED_COLUMNS) - set(df.columns)
    )

    if missing_columns:
        raise ValueError(
            "Colonne mancanti: "
            + ", ".join(missing_columns)
        )


# ==============================================================
# RINOMINA COLONNE
# ==============================================================

COLUMN_RENAME = {
    "CAZI": "mersy_azienda(CAZI)",
    "CFOR": "mersy_fornitore(CFOR)",
    "CEDI": "mersy_cedi(CEDI)",
    "CPCO": "mersy_punto_consegna(CPCO)",

    "CDOCIAA": "mersy_tipo_documento(CDOCIAA)",
    "NDOGIAA": "mersy_n_fattura(NDOGIAA)",
    "TDATIAA": "mersy_data_fattura(TDATIAA)",

    "NDOGIAB": "mersy_n_DDT(NDOGIAB)",
    "TDATIAB": "mersy_data_DDT(TDATIAB)",

    "CART": "mersy_articolo(CART)",
    "XART": "mersy_descrizione_articolo(XART)",
    "CALA": "mersy_articolo_fornitore(CALA)",

    "Qtà": "mersy_qta_fattura(Qtà)",
    "Importo_ns_fattura": "mersy_importo_ns_accettazione(Importo_ns_fattura)",
    "PF02IAL": "mersy_prezzo_fattura(PF02IAL)",
    "QTA ORDINE": "mersy_qta_ordine(QTA ORDINE)",
    "QTA ACCETT": "mersy_qta_accettata(QTA ACCETT)",

    "CUMF": "mersy_um_prezzo(CUMF)",
    "CSTQ": "mersy_stato_quadratura(CSTQ)",
    "CEAS": "mersy_archiviazione_ottica(CEAS)",
}


# ==============================================================
# ESECUZIONE
# ==============================================================

# 1. Lettura della sorgente
df_raw = read_source()

# 2. Applicazione dei tipi con i nomi originali
df_fatture = apply_schema(df_raw)

# 3. Validazione dello schema originale
validate_schema(df_fatture)

# 4. Filtro sul tipo di documento: definisce il perimetro.
#    Viene applicato prima di ogni join.
righe_prima_del_filtro = len(df_fatture)

df_fatture = df_fatture.loc[
    df_fatture["CDOCIAA"].isin(CDOCIAA_FILTRO)
].copy()

print(f"Origine: {CARICHI_PATH}")
print(f"Righe lette: {righe_prima_del_filtro:,}")
print(
    "Righe dopo il filtro CDOCIAA = "
    f"{', '.join(CDOCIAA_FILTRO)}: {len(df_fatture):,}"
)

display(
    df_fatture["CDOCIAA"]
    .value_counts(dropna=False)
    .rename_axis("CDOCIAA")
    .reset_index(name="Righe")
)

# 5. Rinomina finale
df_fatture = df_fatture.rename(
    columns=COLUMN_RENAME
)

print(f"Colonne: {len(df_fatture.columns)}")

display(df_fatture.head())


# ==============================================================
# NOMI DI COLONNA UTILIZZATI NELLE CELLE SUCCESSIVE
# ==============================================================

COL_CAZI = "mersy_azienda(CAZI)"
COL_CFOR = "mersy_fornitore(CFOR)"
COL_CEDI = "mersy_cedi(CEDI)"
COL_CPCO = "mersy_punto_consegna(CPCO)"

COL_CDOCIAA = "mersy_tipo_documento(CDOCIAA)"
COL_NDOGIAA = "mersy_n_fattura(NDOGIAA)"
COL_TDATIAA = "mersy_data_fattura(TDATIAA)"

COL_NDOGIAB = "mersy_n_DDT(NDOGIAB)"
COL_TDATIAB = "mersy_data_DDT(TDATIAB)"

COL_CART = "mersy_articolo(CART)"
COL_XART = "mersy_descrizione_articolo(XART)"
COL_CALA = "mersy_articolo_fornitore(CALA)"

COL_CEAS = "mersy_archiviazione_ottica(CEAS)"
COL_CARF = "mersy_articolo_fornitore(CARF)"
COL_CEAN = "mersy_ean(CEAN)"
COL_SBAR = "mersy_tipo_ean(SBAR)"

# %% [8]


# ==============================================================
# CARICHI - PERIMETRO XML E INDICE
# ==============================================================
# Corrisponde ai passi 4, 5, 6 e 8 di carichi_dataset:
#
#   4  pulizia delle chiavi
#   5  preparazione delle chiavi delle fatture XML
#   6  inner join con le testate XML (perimetro effettivo)
#   8  indice progressivo
#
# L'indice identifica univocamente la riga del carico e viene
# utilizzato dai lookup della query merge, che assegnano a ogni
# riga XML il carico con l'indice più basso.
#
# Viene creato DOPO i filtri, quindi la numerazione riguarda
# soltanto il perimetro effettivamente analizzato.
# ==============================================================

# Chiavi del lato carichi
CHIAVI_CARICHI = [
    COL_CFOR,
    COL_CART,
    COL_NDOGIAA,
    COL_NDOGIAB,
    COL_CEAS,
]


def vuota_in_mancante(serie: pd.Series) -> pd.Series:
    """
    Converte le stringhe vuote in valori mancanti.

    Serve a impedire che una chiave priva di valore generi
    abbinamenti nei join successivi: in pandas, a differenza di
    SQL, due chiavi nulle si incontrano, e due stringhe vuote
    si incontrano anch'esse.

    Le chiavi vengono soltanto rifilate. Se in futuro emergessero
    caratteri non stampabili, qui va applicata una pulizia
    equivalente alla pulizia dei caratteri non stampabili:

        .str.replace(r"[\\x00-\\x1f\\x7f-\\x9f]", "", regex=True)
    """

    return serie.astype("string").str.strip().replace("", pd.NA)


def solo_chiavi_valide(
    df: pd.DataFrame,
    colonne: list,
) -> pd.DataFrame:
    """
    Restituisce le righe che hanno tutte le chiavi indicate
    valorizzate. Usata sulle anagrafiche prima del join: una riga
    con chiave nulla non può agganciare nulla di legittimo.
    """

    return df.loc[df[colonne].notna().all(axis=1)].copy()


# --------------------------------------------------------------
# 1. PULIZIA DELLE CHIAVI DEI CARICHI
# --------------------------------------------------------------

for colonna in CHIAVI_CARICHI:
    df_fatture[colonna] = vuota_in_mancante(df_fatture[colonna])


# --------------------------------------------------------------
# 2. CHIAVI DELLE FATTURE PRESENTI NEGLI XML
# --------------------------------------------------------------
# CEAS coincide con il nome del file XML senza estensione.
# Basta df_xml_fatture: serve soltanto l'elenco distinto delle
# coppie (CEAS, numero fattura), non l'intera testata.

def ceas_da_nome_file(nome_file) -> str:
    """
    Restituisce il CEAS, cioè il nome del file XML senza
    estensione.

    Si usa il solo nome del file, senza l'eventuale percorso
    interno allo ZIP, perché la corrispondenza è con il nome del
    file così come compare nell'archivio.
    """

    nome = os.path.basename(str(nome_file))

    if nome.lower().endswith(".xml"):
        nome = nome[:-4]

    return nome


fatture_xml = (
    df_xml_fatture[["FileXML", "NumeroFattura"]]
    .drop_duplicates()
    .copy()
)

fatture_xml["CEAS_XML"] = (
    fatture_xml["FileXML"]
    .map(ceas_da_nome_file)
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

fatture_xml["NDOGIAA_XML"] = (
    fatture_xml["NumeroFattura"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

# Vengono eliminate le chiavi incomplete e i duplicati,
# per impedire che una testata XML duplicata duplichi i carichi.
fatture_xml = (
    fatture_xml[["CEAS_XML", "NDOGIAA_XML"]]
    .dropna()
    .drop_duplicates()
)

print(
    "Fatture distinte presenti negli XML: "
    f"{len(fatture_xml):,}"
)


# --------------------------------------------------------------
# 3. INNER JOIN: SOLO I CARICHI CON LA FATTURA NEGLI XML
# --------------------------------------------------------------

righe_prima_del_perimetro = len(df_fatture)

df_fatture = df_fatture.merge(
    fatture_xml,
    how="inner",
    left_on=[COL_CEAS, COL_NDOGIAA],
    right_on=["CEAS_XML", "NDOGIAA_XML"],
)

# Le chiavi arrivate dall'XML servivano solo all'abbinamento
df_fatture = df_fatture.drop(
    columns=["CEAS_XML", "NDOGIAA_XML"]
)

print(
    "Righe escluse perché senza fattura XML: "
    f"{righe_prima_del_perimetro - len(df_fatture):,}"
)
print(f"Righe nel perimetro: {len(df_fatture):,}")


# --------------------------------------------------------------
# 4. INDICE PROGRESSIVO
# --------------------------------------------------------------

df_fatture = df_fatture.reset_index(drop=True)

df_fatture["Indice"] = pd.Series(
    range(1, len(df_fatture) + 1),
    dtype="Int64",
)

print(f"Indice assegnato da 1 a {len(df_fatture):,}")

# %% [9]


# ==============================================================
# CARICHI - ANAGRAFICHE
# ==============================================================
# Corrisponde ai passi 7, 9, 10 e 12 di carichi_dataset:
#
#   7   merge con l'anagrafica dei punti di consegna
#   9   merge con l'anagrafica articoli (EAN e codice fornitore)
#   12  sostituzione della CALA originale con il CARF
#
# La sostituzione NON applica alcuna normalizzazione: la
# normalizzazione dei codici è governata dal dizionario e avverrà
# in seguito, con la regola prevista per il lato carichi.
# ==============================================================


# ==============================================================
# 1. ANAGRAFICA PUNTI DI CONSEGNA
# ==============================================================

df_cpco = pd.read_excel(
    CPCO_PATH,
    sheet_name=CPCO_SHEET,
    engine="openpyxl",
    usecols=["CPCO", "XPCO"],
    dtype=object,
)

df_cpco["CPCO"] = pd.to_numeric(
    df_cpco["CPCO"],
    errors="coerce",
).astype("Int64")

df_cpco["XPCO"] = (
    df_cpco["XPCO"]
    .astype("string")
    .str.strip()
)

# Un CPCO non deve comparire due volte, altrimenti il join
# duplicherebbe i carichi
cpco_duplicati = (
    df_cpco["CPCO"]
    .dropna()
    .duplicated(keep=False)
)

if cpco_duplicati.any():
    raise ValueError(
        "CPCO duplicati nell'anagrafica punti di consegna: "
        + ", ".join(
            str(valore)
            for valore in df_cpco.loc[
                cpco_duplicati, "CPCO"
            ].unique()[:10]
        )
    )

df_cpco = df_cpco.rename(
    columns={"XPCO": "mersy_descrizione_punto_consegna(XPCO)"}
)

# Le righe con chiave nulla non possono agganciare nulla
df_cpco = solo_chiavi_valide(df_cpco, ["CPCO"])

righe_prima = len(df_fatture)

df_fatture = df_fatture.merge(
    df_cpco,
    how="left",
    left_on=COL_CPCO,
    right_on="CPCO",
    validate="many_to_one",
)

df_fatture = df_fatture.drop(columns=["CPCO"])

assert len(df_fatture) == righe_prima, (
    "Il join CPCO ha modificato il numero di righe"
)

print("Colonne aggiunte: mersy_descrizione_punto_consegna(XPCO)")


# ==============================================================
# 2. ANAGRAFICA EAN
# ==============================================================

df_ean = pd.read_excel(
    EAN_PATH,
    sheet_name=EAN_SHEET,
    engine="openpyxl",
    usecols=["CART", "SBAR", "CEAN"],
    dtype={
        "CART": "string",
        "SBAR": "string",
        "CEAN": "string",
    },
)

for colonna in ["CART", "SBAR", "CEAN"]:
    df_ean[colonna] = (
        df_ean[colonna]
        .astype("string")
        .str.strip()
    )

# Verifica: un solo record EAN per ogni CART
ean_duplicati = df_ean["CART"].duplicated(keep=False)

if ean_duplicati.any():
    raise ValueError(
        "CART duplicati nell'anagrafica EAN: "
        + ", ".join(
            df_ean.loc[ean_duplicati, "CART"]
            .dropna()
            .unique()[:10]
        )
    )

df_ean = df_ean.rename(
    columns={
        "SBAR": COL_SBAR,
        "CEAN": COL_CEAN,
    }
)

# Le righe con chiave nulla non possono agganciare nulla
df_ean = solo_chiavi_valide(df_ean, ["CART"])

righe_prima = len(df_fatture)

df_fatture = df_fatture.merge(
    df_ean,
    how="left",
    left_on=COL_CART,
    right_on="CART",
    validate="many_to_one",
)

df_fatture = df_fatture.drop(columns=["CART"])

assert len(df_fatture) == righe_prima, (
    "Il join EAN ha modificato il numero di righe"
)

print(f"Colonne aggiunte: {COL_SBAR}, {COL_CEAN}")


# ==============================================================
# 3. ANAGRAFICA CODICI ARTICOLO FORNITORE
# ==============================================================
# Chiave: CART + CFOR.
#
# Il codice articolo del fornitore dipende dal fornitore, quindi
# la deduplica è per coppia e non per solo CART: raggruppando sul
# solo CART, più fornitori sullo stesso articolo finirebbero per
# condividere un codice unico e il match CALA fallirebbe
# sistematicamente per uno dei due.
#
# In caso di duplicati vale la riga con TDEC più recente.

df_carf = pd.read_excel(
    CARF_PATH,
    sheet_name=CARF_SHEET,
    engine="openpyxl",
    usecols=["CART", "CFOR", "TDEC", "CARF", "XARF"],
    dtype={
        "CART": "string",
        "CFOR": "string",
        "CARF": "string",
        "XARF": "string",
    },
)

for colonna in ["CART", "CFOR", "CARF", "XARF"]:
    df_carf[colonna] = (
        df_carf[colonna]
        .astype("string")
        .str.strip()
    )

df_carf["TDEC"] = pd.to_datetime(
    df_carf["TDEC"],
    errors="coerce",
)

# Per ogni coppia CART + CFOR conserva la riga con TDEC più
# recente. Le date mancanti vengono considerate meno recenti.
df_carf = (
    df_carf
    .sort_values(
        by="TDEC",
        ascending=False,
        na_position="last",
    )
    .drop_duplicates(
        subset=["CART", "CFOR"],
        keep="first",
    )
)

df_carf = df_carf.rename(
    columns={
        "CARF": COL_CARF,
        "XARF": "mersy_descrizione_fornitore(XARF)",
    }
)

# TDEC non deve entrare nel DataFrame finale
df_carf = df_carf.drop(columns=["TDEC"])

# Le righe con chiave nulla non possono agganciare nulla
df_carf = solo_chiavi_valide(df_carf, ["CART", "CFOR"])

righe_prima = len(df_fatture)

df_fatture = df_fatture.merge(
    df_carf,
    how="left",
    left_on=[COL_CART, COL_CFOR],
    right_on=["CART", "CFOR"],
    validate="many_to_one",
)

df_fatture = df_fatture.drop(columns=["CART", "CFOR"])

assert len(df_fatture) == righe_prima, (
    "Il join CARF ha modificato il numero di righe"
)

print(
    "Colonne aggiunte: "
    f"{COL_CARF}, mersy_descrizione_fornitore(XARF)"
)


# ==============================================================
# 4. SOSTITUZIONE DELLA CALA ORIGINALE CON IL CARF
# ==============================================================
# Nessuna normalizzazione: il valore viene portato avanti così
# com'è. La normalizzazione dei codici è governata dal dizionario
# e verrà applicata in seguito.

df_fatture = df_fatture.drop(columns=[COL_CALA])

df_fatture = df_fatture.rename(
    columns={COL_CARF: COL_CALA}
)

# Da questo momento COL_CARF non esiste più: la colonna che
# contiene il codice articolo del fornitore è COL_CALA.
print(f"CALA ricostruita da CARF ({COL_CALA})")

# %% [10]


# ==============================================================
# CARICHI - CHIAVI DI CONFRONTO (DDT e codici articolo)
# ==============================================================
# Qui si costruiscono tutte le chiavi normalizzate del lato
# carichi: quella del DDT e quelle dei tre codici articolo su cui
# avviene l'abbinamento.
#
#     NDOGIAB  -> NDOGIAB_Chiave    (foglio RegoleDDT)
#     CEAN     -> CEAN_Chiave       (foglio RegoleCodiciCarico, EAN)
#     CART     -> CART_Chiave       (foglio RegoleCodiciCarico, CODICE_CLIENTE)
#     CALA     -> CALA_Chiave       (foglio RegoleCodiciCarico, CODICE_FORNITORE)
#
# È il punto in cui il dizionario lavora davvero, e lavora su
# entrambi i lati: la regola viene letta per singolo fornitore.
# In assenza di una riga si applica il default (RIMUOVI_SEPARATORI).
#
# I valori originali restano nelle colonne di partenza: la
# normalizzazione serve solo al confronto, mentre il report
# continua a mostrare i codici reali.
# ==============================================================

# Le chiavi distinte vengono normalizzate una sola volta
valori_ddt = (
    df_fatture[[COL_CFOR, COL_NDOGIAB]]
    .drop_duplicates()
    .copy()
)

valori_ddt["NDOGIAB_Chiave"] = [
    normalizza_ddt_carico(dizionari, cfor, valore)
    for cfor, valore in zip(
        valori_ddt[COL_CFOR],
        valori_ddt[COL_NDOGIAB],
    )
]

valori_ddt["NDOGIAB_Chiave"] = (
    valori_ddt["NDOGIAB_Chiave"]
    .astype("string")
)

righe_prima = len(df_fatture)

df_fatture = df_fatture.merge(
    valori_ddt,
    how="left",
    on=[COL_CFOR, COL_NDOGIAB],
    validate="many_to_one",
)

assert len(df_fatture) == righe_prima, (
    "Il join della chiave DDT ha modificato il numero di righe"
)


# --------------------------------------------------------------
# RIEPILOGO DELLE REGOLE APPLICATE
# --------------------------------------------------------------

regole_applicate = (
    valori_ddt[COL_CFOR]
    .map(
        lambda cfor: (
            regola_ddt_carico(dizionari, cfor)
            or "RIMUOVI_SEPARATORI (default)"
        )
    )
)

riepilogo_regole = (
    pd.DataFrame({
        "Regola": regole_applicate,
        "CFOR": valori_ddt[COL_CFOR],
    })
    .groupby("Regola", dropna=False)
    .agg(
        Fornitori=("CFOR", "nunique"),
    )
    .reset_index()
    .sort_values(
        "Fornitori",
        ascending=False,
    )
    .reset_index(drop=True)
)

print("Regole DDT applicate al lato carichi:")
display(riepilogo_regole)

print(
    "Righe senza chiave DDT (NDOGIAB non valorizzato): "
    f"{int(df_fatture['NDOGIAB_Chiave'].isna().sum()):,}"
)
print(f"Righe totali: {len(df_fatture):,}")


# --------------------------------------------------------------
# CHIAVI DEI CODICI ARTICOLO
# --------------------------------------------------------------
# Stesse regole del lato XML, lette dal foglio RegoleCodiciCarico.
# Le chiavi distinte vengono normalizzate una sola volta e poi
# riportate nel DataFrame, come già fatto per NDOGIAB_Chiave.

CODICI_CARICO = [
    (COL_CEAN, "EAN", "CEAN_Chiave"),
    (COL_CART, "CODICE_CLIENTE", "CART_Chiave"),
    (COL_CALA, "CODICE_FORNITORE", "CALA_Chiave"),
]

for colonna_carico, campo_canonico, colonna_chiave in CODICI_CARICO:

    valori_codice = (
        df_fatture[[COL_CFOR, colonna_carico]]
        .drop_duplicates()
        .copy()
    )

    valori_codice[colonna_chiave] = [
        normalizza_codice_carico(dizionari, cfor, campo_canonico, valore)
        for cfor, valore in zip(
            valori_codice[COL_CFOR],
            valori_codice[colonna_carico],
        )
    ]

    valori_codice[colonna_chiave] = (
        valori_codice[colonna_chiave].astype("string")
    )

    righe_prima = len(df_fatture)

    df_fatture = df_fatture.merge(
        valori_codice,
        how="left",
        on=[COL_CFOR, colonna_carico],
        validate="many_to_one",
    )

    assert len(df_fatture) == righe_prima, (
        f"Il join della chiave {campo_canonico} ha modificato il numero di righe"
    )


# --------------------------------------------------------------
# RIEPILOGO E DIAGNOSTICA DEI CODICI
# --------------------------------------------------------------

CAMPI_CANONICI_ATTESI = [
    "EAN",
    "CODICE_CLIENTE",
    "CODICE_FORNITORE",
]

righe_riepilogo = []

for colonna_carico, campo_canonico, colonna_chiave in CODICI_CARICO:

    valori = df_fatture[colonna_carico].dropna()
    chiavi = df_fatture[colonna_chiave].dropna()

    modificati = (
        df_fatture[[colonna_carico, colonna_chiave]]
        .dropna()
        .drop_duplicates()
        .query(f"`{colonna_carico}` != `{colonna_chiave}`")
    )

    righe_riepilogo.append({
        "CampoCanonico": campo_canonico,
        "ColonnaCarico": colonna_carico,
        "ValoriDistinti": int(valori.nunique()),
        "ChiaviDistinte": int(chiavi.nunique()),
        "ValoriModificati": len(modificati),
    })

print("Chiavi dei codici articolo sul lato carichi:")
display(pd.DataFrame(righe_riepilogo))

# Un CampoCanonico scritto in modo diverso da quelli attesi non
# viene mai applicato: la regola resta invisibile. Qui si
# elencano, così un refuso nel dizionario non passa inosservato.
campi_non_riconosciuti = sorted({
    campo
    for _, campo in dizionari["CodiciCarico"]
    if campo not in CAMPI_CANONICI_ATTESI
})

if campi_non_riconosciuti:
    print(
        "ATTENZIONE - CampoCanonico non riconosciuto nel foglio "
        "RegoleCodiciCarico: le relative regole non vengono applicate."
    )
    display(pd.DataFrame({
        "CampoCanonico": campi_non_riconosciuti,
        "Atteso": " / ".join(CAMPI_CANONICI_ATTESI),
    }))


# --------------------------------------------------------------
# COERENZA DELLE REGOLE FRA I DUE LATI
# --------------------------------------------------------------
# I due lati hanno fogli distinti (RegoleCodiciXML e
# RegoleCodiciCarico) e leggono la regola in modo diverso:
#
#     lato XML      (CFOR, CodiceTipoXML)     -> piu' tipi possono
#                                               finire sullo stesso
#                                               CampoCanonico
#     lato carichi  (CFOR, CampoCanonico)
#
# Quando una regola manca si applica REGOLA_DEFAULT, quindi una
# riga assente NON equivale a "NESSUNA_REGOLA": il valore viene
# comunque ripulito e, se contiene separatori, diventa diverso da
# quello che ci si aspetta. Lo stesso codice scritto identico sui
# due lati finisce per produrre due chiavi diverse e la riga non
# si abbina, senza alcun errore.
#
# Qui si confrontano le regole effettivamente applicate dai due
# lati. L'elenco e' puramente diagnostico: non cambia nulla nel
# risultato, ma rende visibile un disallineamento nel dizionario.

def regola_effettiva(regola):
    """
    La regola che il notebook applica davvero, default compreso.

    Una regola assente o non riconosciuta non blocca nulla: si
    ricade su REGOLA_DEFAULT, esattamente come fa normalizza().
    """

    regola_pulita = regola_normalizzata(regola)

    if regola_pulita not in REGOLE:
        return REGOLA_DEFAULT

    return regola_pulita


def testo_regola(regola):
    """Etichetta della regola, con il default dichiarato."""

    if regola_normalizzata(regola) is None:
        return f"{REGOLA_DEFAULT} (default)"

    return regola_effettiva(regola)


# Il confronto avviene sulle regole EFFETTIVE: una regola
# inesistente e una non riconosciuta producono lo stesso
# risultato, e non devono comparire come disallineamento. Il
# testo mostrato resta quello scritto nel foglio.
def coppia_regola(regola):
    return regola_effettiva(regola), testo_regola(regola)


# Lato XML: un CampoCanonico puo' raccogliere piu' CodiceTipoXML,
# quindi si raccoglie l'insieme delle regole applicate.
regole_xml_campo = {}

for (cfor, tipo), (campo, regola) in dizionari["CodiciXML"].items():

    if campo is None:
        continue

    regole_xml_campo.setdefault((cfor, campo), []).append(coppia_regola(regola))

# Lato carichi: la regola e' gia' per CampoCanonico
regole_carico_campo = {
    (cfor, campo): coppia_regola(regola)
    for (cfor, campo), regola in dizionari["CodiciCarico"].items()
}

regole_non_allineate = []

for (cfor, campo), coppie in sorted(regole_xml_campo.items()):

    regola_carico, testo_carico = regole_carico_campo.get(
        (cfor, campo),
        (REGOLA_DEFAULT, f"{REGOLA_DEFAULT} (default)"),
    )

    # Stessa regola effettiva sui due lati: nessun problema.
    if {effettiva for effettiva, _ in coppie} == {regola_carico}:
        continue

    regole_non_allineate.append({
        "CFOR": cfor,
        "Campo": campo,
        "RegolaXML": " / ".join(sorted({testo for _, testo in coppie})),
        "RegolaCarico": testo_carico,
    })

# Stesso controllo sui DDT, dove le due regole stanno sulla stessa
# riga e quindi il disallineamento e' piu' facile da correggere.
for cfor, (regola_xml, regola_carico) in sorted(dizionari["DDT"].items()):

    if regola_effettiva(regola_xml) == regola_effettiva(regola_carico):
        continue

    testo_xml = testo_regola(regola_xml)
    testo_carico = testo_regola(regola_carico)

    regole_non_allineate.append({
        "CFOR": cfor,
        "Campo": "DDT",
        "RegolaXML": testo_xml,
        "RegolaCarico": testo_carico,
    })

regole_non_allineate = pd.DataFrame(
    regole_non_allineate,
    columns=["CFOR", "Campo", "RegolaXML", "RegolaCarico"],
)

if len(regole_non_allineate):

    print(
        "ATTENZIONE - regole diverse fra i due lati: lo stesso codice"
        " scritto identico produce chiavi diverse e non si abbina."
        " Allineare le regole nei fogli RegoleCodiciXML /"
        " RegoleCodiciCarico (o RegoleDDT)."
    )
    display(regole_non_allineate)
else:
    print("Regole dei due lati allineate.")


# %% [11]


# ==============================================================
# XML - FORNITORE E CHIAVE DDT
# ==============================================================
# Controparte della cella [10] sul lato XML.
#
# L'XML non contiene il codice fornitore: CFOR si ricava dai
# carichi attraverso la coppia CEAS + NDOGIAA, la stessa su cui
# la cella [8] ha definito il perimetro.
#
#     FileXML        →  CEAS      (nome del file senza .xml)
#     NumeroFattura  →  NDOGIAA
#
# Una volta disponibile il CFOR, NumeroDDT_Chiave si calcola con
# la regola del foglio RegoleDDT (colonna RegolaDDTXML). Se il
# fornitore non è configurato, o se la riga XML non rientra nel
# perimetro, si applica la regola di default RIMUOVI_SEPARATORI.
#
# NumeroDDT resta il valore originale.
# ==============================================================

COL_CFOR_XML = "CFOR_XML"

COL_CHIAVE_RIGA_XML = "ChiaveRigaXML"


# --------------------------------------------------------------
# 1. CHIAVI XML: CEAS E NUMERO FATTURA
# --------------------------------------------------------------
# Stesse regole della cella [8]: il CEAS è il nome del file
# senza estensione, il numero fattura è quello del body.

df_xml_fatture["CEAS_XML"] = (
    df_xml_fatture["FileXML"]
    .map(ceas_da_nome_file)
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

df_xml_fatture["NDOGIAA_XML"] = (
    df_xml_fatture["NumeroFattura"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)


# --------------------------------------------------------------
# 2. MAPPA CEAS + NDOGIAA → CFOR
# --------------------------------------------------------------
# Nasce da df_fatture, quindi soltanto dalle fatture che hanno
# un XML. Le righe XML fuori perimetro resteranno senza CFOR.

mappa_cfor = (
    df_fatture[[COL_CEAS, COL_NDOGIAA, COL_CFOR]]
    .drop_duplicates()
    .copy()
)

# Una fattura con più di un fornitore non è attesa: viene
# segnalata e si tiene il primo CFOR in ordine alfabetico.
fornitori_per_fattura = (
    mappa_cfor
    .groupby([COL_CEAS, COL_NDOGIAA], dropna=False)[COL_CFOR]
    .nunique(dropna=False)
)

fatture_ambiguity = fornitori_per_fattura[fornitori_per_fattura > 1]

if len(fatture_ambiguity) > 0:
    print(
        "ATTENZIONE - fatture con più di un CFOR: "
        f"{len(fatture_ambiguity):,}"
    )
    display(
        fatture_ambiguity
        .reset_index(name="CFOR distinti")
        .head(20)
    )

mappa_cfor = (
    mappa_cfor
    .sort_values(
        [COL_CEAS, COL_NDOGIAA, COL_CFOR],
        na_position="last",
    )
    .drop_duplicates(
        subset=[COL_CEAS, COL_NDOGIAA],
        keep="first",
    )
)

# Chiavi incomplete: nessun abbinamento su null
mappa_cfor = solo_chiavi_valide(mappa_cfor, [COL_CEAS, COL_NDOGIAA])

print(f"Fatture distinte nella mappa CFOR: {len(mappa_cfor):,}")


# --------------------------------------------------------------
# 3. CFOR SUL LATO XML
# --------------------------------------------------------------

righe_prima = len(df_xml_fatture)

df_xml_fatture = df_xml_fatture.merge(
    mappa_cfor,
    how="left",
    left_on=["CEAS_XML", "NDOGIAA_XML"],
    right_on=[COL_CEAS, COL_NDOGIAA],
    validate="many_to_one",
)

assert len(df_xml_fatture) == righe_prima, (
    "Il join del CFOR ha modificato il numero di righe XML"
)

df_xml_fatture = df_xml_fatture.drop(
    columns=[COL_CEAS, COL_NDOGIAA]
)

df_xml_fatture = df_xml_fatture.rename(
    columns={COL_CFOR: COL_CFOR_XML}
)


# --------------------------------------------------------------
# 4. NUMERO DDT NORMALIZZATO
# --------------------------------------------------------------
# Come nella cella [10]: le chiavi distinte vengono normalizzate
# una sola volta, perché la normalizzazione è costosa.

valori_ddt_xml = (
    df_xml_fatture[[COL_CFOR_XML, "NumeroDDT"]]
    .drop_duplicates()
    .copy()
)

valori_ddt_xml["NumeroDDT_Chiave"] = [
    normalizza_ddt_xml(dizionari, cfor, valore)
    for cfor, valore in zip(
        valori_ddt_xml[COL_CFOR_XML],
        valori_ddt_xml["NumeroDDT"],
    )
]

valori_ddt_xml["NumeroDDT_Chiave"] = (
    valori_ddt_xml["NumeroDDT_Chiave"].astype("string")
)

righe_prima = len(df_xml_fatture)

df_xml_fatture = df_xml_fatture.merge(
    valori_ddt_xml,
    how="left",
    on=[COL_CFOR_XML, "NumeroDDT"],
    validate="many_to_one",
)

assert len(df_xml_fatture) == righe_prima, (
    "Il join della chiave DDT ha modificato il numero di righe XML"
)


# --------------------------------------------------------------
# 5. RIEPILOGO
# --------------------------------------------------------------

righe_senza_cfor = int(df_xml_fatture[COL_CFOR_XML].isna().sum())

print(
    "Righe XML senza CFOR (fuori perimetro o fornitore assente): "
    f"{righe_senza_cfor:,} su {len(df_xml_fatture):,}"
)

print(
    "Righe XML senza chiave DDT (NumeroDDT non valorizzato): "
    f"{int(df_xml_fatture['NumeroDDT_Chiave'].isna().sum()):,}"
)

regole_xml = (
    pd.DataFrame(
        {
            "Regola": [
                regola_ddt_xml(dizionari, cfor)
                or "RIMUOVI_SEPARATORI (default)"
                for cfor in valori_ddt_xml[COL_CFOR_XML]
            ],
            "CFOR": valori_ddt_xml[COL_CFOR_XML],
        }
    )
    .groupby("Regola", dropna=False)
    .agg(
        Fornitori=("CFOR", "nunique"),
        ValoriDDT=("CFOR", "size"),
    )
    .reset_index()
    .sort_values("Fornitori", ascending=False)
    .reset_index(drop=True)
)

print("Regole DDT applicate al lato XML:")
display(regole_xml)


# %% [12]


# ==============================================================
# XML - CODICI ARTICOLO NORMALIZZATI
# ==============================================================
# df_xml_codici_articolo è in formato lungo: una riga per codice.
# La cella aggiunge tre colonne:
#
#     CFOR_XML                   fornitore, preso dalla riga XML
#     CampoCanonico              destinazione del codice
#                                (EAN, CART, CALA, ...)
#     CodiceValoreNormalizzato   valore pronto per il confronto
#
# La corrispondenza su CodiceTipoXML non distingue maiuscole e
# minuscole.
#
# REGOLA DEI TIPI CODIFICATI - nella tabella restano soltanto i
# codici il cui CodiceTipoXML compare nel foglio RegoleCodiciXML
# per quel CFOR. Un tipo non codificato non può comunque
# contribuire alla chiave articolo (finirebbe in NON_MAPPATO|
# tipo, che la gerarchia non guarda): tenerlo nel foglio di
# lavoro non serve a nulla.
#
# Conseguenza voluta: un tipo nuovo partecipa SOLO se qualcuno ne
# scrive la riga nel dizionario. Abilitarlo è una riga in più
# nell'Excel, non una modifica al codice.
#
# I tipi virtuali costruiti dal notebook sono due: "Descrizione"
# (aggiunta in questa cella, passo 2) e RiferimentoAmministrazione
# (aggiunto dalla cella [3]).
#
# NOTA - RiferimentoAmministrazione è il tipo virtuale costruito
# dalla cella [3]: nell'XML non è un CodiceArticolo. Segue la
# stessa regola di tutti gli altri tipi.
# ==============================================================


# --------------------------------------------------------------
# 1. CFOR SULLA RIGA DI CODICE
# --------------------------------------------------------------

assert df_xml_fatture[COL_CHIAVE_RIGA_XML].is_unique, (
    "ChiaveRigaXML non è unica: il CFOR non può essere attribuito"
)

df_xml_codici_articolo = df_xml_codici_articolo.merge(
    df_xml_fatture[[COL_CHIAVE_RIGA_XML, COL_CFOR_XML]],
    how="left",
    on=COL_CHIAVE_RIGA_XML,
    validate="many_to_one",
)


# --------------------------------------------------------------
# 2. DESCRIZIONE COME CODICE ARTICOLO
# --------------------------------------------------------------
# Alcuni fornitori scrivono il codice articolo dentro la
# descrizione invece che in un CodiceArticolo: "1234 melatonina
# in compresse 1mg" con il codice cliente 1234.
#
# La descrizione viene proposta come codice di tipo
# "Descrizione". Perché una riga venga aggiunta servono due cose:
#
#   1. il dizionario deve codificare il tipo "Descrizione" per
#      quel CFOR: altrimenti il codice viene comunque scartato
#      dal filtro del passo 5 e la riga sarebbe inutile;
#   2. la linea non deve avere NESSUN codice codificato.
#
# Il punto 2 è la ragione per cui questo passo sta qui e non
# nella cella [3]: "codificato" dipende dal dizionario. In [3]
# l'unica cosa verificabile era "la linea non ha nessun
# CodiceArticolo", e una linea con un codice di tipo che il
# dizionario non conosce (per esempio MINSAN) restava senza
# descrizione pur non avendo, di fatto, nessun codice utilizzabile.
#
# Perché la linea non deve avere codici codificati: il frammento
# di descrizione finirebbe nello stesso campo canonico del codice
# vero e nel report comparirebbe al suo posto. Una linea con
# codice cliente C002 e descrizione "BURRO" mostrerebbe
# "Codice Art. Fornitore = BURRO".
#
# L'ordine di priorità resta quello dei tipi veri: la descrizione
# si aggiunge solo dove non c'è nulla di meglio.

codici_codificati = set(dizionari["CodiciXML"].keys())


def codice_tipo_codificato(cfor, codice_tipo) -> bool:
    """
    Vero se il tipo di codice è configurato nel foglio
    RegoleCodiciXML per quel fornitore.
    """

    return (chiave_cfor(cfor), chiave_testo(codice_tipo)) in codici_codificati


# Fornitori che hanno davvero la riga "Descrizione" nel dizionario:
# per tutti gli altri la riga non viene nemmeno costruita, invece di
# costruirla e scartarla al passo 5
fornitori_con_descrizione = {
    cfor
    for cfor, tipo in codici_codificati
    if tipo == chiave_testo("Descrizione")
}

righe_con_codice_codificato = set(
    df_xml_codici_articolo.loc[
        [
            codice_tipo_codificato(cfor, tipo)
            for cfor, tipo in zip(
                df_xml_codici_articolo[COL_CFOR_XML],
                df_xml_codici_articolo["CodiceTipoXML"],
            )
        ],
        COL_CHIAVE_RIGA_XML,
    ]
)

descrizioni_da_aggiungere = df_xml_fatture.loc[
    df_xml_fatture[COL_CFOR_XML].map(chiave_cfor).isin(
        fornitori_con_descrizione
    )
    & ~df_xml_fatture[COL_CHIAVE_RIGA_XML].isin(righe_con_codice_codificato)
    & df_xml_fatture["Descrizione"].notna()
    & df_xml_fatture["Descrizione"].astype("string").str.strip().ne("")
]

# Progressivo dopo gli eventuali codici non codificati già
# presenti sulla linea, che restano nella tabella fino al passo 5
progressivo_massimo = (
    df_xml_codici_articolo
    .groupby(COL_CHIAVE_RIGA_XML)["ProgressivoCodiceXML"]
    .max()
)

righe_descrizione = pd.DataFrame({
    COL_CHIAVE_RIGA_XML: descrizioni_da_aggiungere[COL_CHIAVE_RIGA_XML],
    "ProgressivoCodiceXML": (
        pd.to_numeric(
            descrizioni_da_aggiungere[COL_CHIAVE_RIGA_XML].map(progressivo_massimo),
            errors="coerce",
        )
        .fillna(0)
        .astype(int)
        + 1
    ),
    "CodiceTipoXML": "Descrizione",
    "CodiceValoreXML": descrizioni_da_aggiungere["Descrizione"],
    COL_CFOR_XML: descrizioni_da_aggiungere[COL_CFOR_XML],
})

righe_prima_della_descrizione = len(df_xml_codici_articolo)

if not righe_descrizione.empty:

    df_xml_codici_articolo = pd.concat(
        [df_xml_codici_articolo, righe_descrizione],
        ignore_index=True,
    )

assert len(df_xml_codici_articolo) == (
    righe_prima_della_descrizione + len(righe_descrizione)
), "L'aggiunta della descrizione ha modificato il numero di righe"

print(
    "Righe di tipo Descrizione aggiunte: "
    f"{len(righe_descrizione):,} (sulle linee senza codici codificati)"
)


# --------------------------------------------------------------
# 3. NORMALIZZAZIONE SULLE TERNE DISTINTE
# --------------------------------------------------------------

valori_codici = (
    df_xml_codici_articolo[
        [COL_CFOR_XML, "CodiceTipoXML", "CodiceValoreXML"]
    ]
    .drop_duplicates()
    .copy()
)

risultati = [
    normalizza_codice_xml(dizionari, cfor, tipo, valore)
    for cfor, tipo, valore in zip(
        valori_codici[COL_CFOR_XML],
        valori_codici["CodiceTipoXML"],
        valori_codici["CodiceValoreXML"],
    )
]

valori_codici["CampoCanonico"] = pd.Series(
    [campo for campo, _ in risultati],
    index=valori_codici.index,
    dtype="string",
)

valori_codici["CodiceValoreNormalizzato"] = pd.Series(
    [valore for _, valore in risultati],
    index=valori_codici.index,
    dtype="string",
)

# Solo per il riepilogo: la regola effettivamente applicata
valori_codici["Regola"] = pd.Series(
    [
        regola_normalizzata(
            regola_codice_xml(dizionari, cfor, tipo)[1]
        )
        or "RIMUOVI_SEPARATORI (default)"
        for cfor, tipo in zip(
            valori_codici[COL_CFOR_XML],
            valori_codici["CodiceTipoXML"],
        )
    ],
    index=valori_codici.index,
    dtype="string",
)


# --------------------------------------------------------------
# 4. RIENTRO NEL DATAFRAME LUNGO
# --------------------------------------------------------------

righe_prima = len(df_xml_codici_articolo)

df_xml_codici_articolo = df_xml_codici_articolo.merge(
    valori_codici[
        [
            COL_CFOR_XML,
            "CodiceTipoXML",
            "CodiceValoreXML",
            "CampoCanonico",
            "CodiceValoreNormalizzato",
        ]
    ],
    how="left",
    on=[COL_CFOR_XML, "CodiceTipoXML", "CodiceValoreXML"],
    validate="many_to_one",
)

assert len(df_xml_codici_articolo) == righe_prima, (
    "Il join dei codici normalizzati ha modificato il numero di righe"
)


# --------------------------------------------------------------
# 5. SOLO I TIPI CODIFICATI NEL DIZIONARIO
# --------------------------------------------------------------
# Il conteggio si fa PRIMA del filtro: è l'unico momento in cui
# si può ancora sapere quanti codici sono stati messi da parte.
# Dopo il filtro un tipo non codificato semplicemente non esiste
# più, e non ha senso mostrarlo in una tabella.

codici_prima_del_filtro = len(df_xml_codici_articolo)

maschera_codificata = pd.Series(
    [
        codice_tipo_codificato(cfor, tipo)
        for cfor, tipo in zip(
            df_xml_codici_articolo[COL_CFOR_XML],
            df_xml_codici_articolo["CodiceTipoXML"],
        )
    ],
    index=df_xml_codici_articolo.index,
    dtype=bool,
)

df_xml_codici_articolo = (
    df_xml_codici_articolo.loc[maschera_codificata].copy()
)

print(
    "Codici esclusi perché il tipo non è nel dizionario: "
    f"{codici_prima_del_filtro - len(df_xml_codici_articolo):,}"
    f" su {codici_prima_del_filtro:,}"
)


# --------------------------------------------------------------
# 6. RIEPILOGO
# --------------------------------------------------------------

senza_valore = int(
    df_xml_codici_articolo["CodiceValoreNormalizzato"].isna().sum()
)

print(f"Codici senza valore normalizzato: {senza_valore:,}")

riepilogo_codici = (
    valori_codici
    .groupby(["Regola", "CampoCanonico"], dropna=False)
    .agg(
        ValoriDistinti=("CodiceValoreXML", "size"),
        CodiciTipo=("CodiceTipoXML", "nunique"),
    )
    .reset_index()
    .sort_values("ValoriDistinti", ascending=False)
    .reset_index(drop=True)
)

print("Regole applicate ai codici articolo XML:")
display(riepilogo_codici)

righe_riferimento = int(
    (
        df_xml_codici_articolo["CodiceTipoXML"]
        == "RiferimentoAmministrazione"
    ).sum()
)

print(
    "Righe di tipo RiferimentoAmministrazione codificate: "
    f"{righe_riferimento:,}"
)


# %% [13]


# ==============================================================
# XML - RIGHE AGGREGATE  (xml_righe_(3))
# ==============================================================
# Trasforma la tabella lunga dei codici in una riga per
# articolo:
#
#   1. righe senza codice → codice tecnico __NESSUN_CODICE
#   2. campo di pivot     → CampoCanonico, o NON_MAPPATO|tipo
#   3. pivot              → una colonna per campo canonico
#   4. ChiaveArticoloXML  → gerarchia EAN → CART → FORN → natura
#   5. aggregazione       → file + fattura + DDT + articolo
#
# Il risultato è df_xml_righe: la granularità diventa finalmente
# confrontabile con quella dei carichi (una riga per articolo
# dentro un DDT), mentre df_xml_fatture resta una riga per linea.
#
# Colonna CFOR: qui si chiama semplicemente CFOR, come in Power
# Query, perché è la controparte di mersy_fornitore(CFOR) dei
# carichi. Nelle celle precedenti era CFOR_XML.
# ==============================================================

NOME_CAMPO_EAN = "EAN"
NOME_CAMPO_FORNITORE = "Codice Art. Fornitore"
NOME_CAMPO_CLIENTE = "Codice Art. Cliente"

TOKEN_NESSUN_CODICE = "__NESSUN_CODICE"

# Valori ammessi nel dizionario → nome della colonna di pivot
CAMPI_CANONICI_PIVOT = {
    "EAN": NOME_CAMPO_EAN,
    "CODICE_FORNITORE": NOME_CAMPO_FORNITORE,
    "CODICE_CLIENTE": NOME_CAMPO_CLIENTE,
}

# Natura della riga, quando nessun codice è disponibile
NATURA_RIGA = {
    "AB": "ABBUONO",
    "AC": "SPESA ACCESSORIA",
    "PR": "PREMIO",
    "SC": "SCONTO",
}


# --------------------------------------------------------------
# 1. RIGHE PRIVE DI CODICE ARTICOLO
# --------------------------------------------------------------
# A queste righe viene assegnato il codice tecnico
# __NESSUN_CODICE, così sopravvivono al pivot. Senza di esse
# le righe descrittive (AB, AC, PR, SC) e tutte le linee senza
# CodiceArticolo sparirebbero dall'aggregazione.

chiavi_con_codice = set(
    df_xml_codici_articolo[COL_CHIAVE_RIGA_XML]
)

righe_senza_codice = (
    df_xml_fatture.loc[
        ~df_xml_fatture[COL_CHIAVE_RIGA_XML].isin(chiavi_con_codice),
        [COL_CHIAVE_RIGA_XML, COL_CFOR_XML],
    ]
    .copy()
)

righe_senza_codice["ProgressivoCodiceXML"] = 0
righe_senza_codice["CodiceTipoXML"] = TOKEN_NESSUN_CODICE
righe_senza_codice["CodiceValoreXML"] = "1"
righe_senza_codice["CampoCanonico"] = pd.NA
righe_senza_codice["CodiceValoreNormalizzato"] = "1"

df_xml_codici = pd.concat(
    [df_xml_codici_articolo, righe_senza_codice],
    ignore_index=True,
)

print(
    "Righe XML senza codice articolo: "
    f"{len(righe_senza_codice):,}"
)


# --------------------------------------------------------------
# 2. CAMPO DI PIVOT
# --------------------------------------------------------------
# Un codice non mappato dal dizionario finisce in una colonna
# propria NON_MAPPATO|tipo: resta visibile senza inquinare i
# campi canonici.

def campo_pivot(codice_tipo, campo_canonico) -> str:
    """
    Colonna di pivot del codice.

    Il ramo NON_MAPPATO|tipo è una rete di sicurezza: dopo il
    filtro della cella [12] i tipi non codificati non arrivano
    più fin qui. Resta nel codice perché la gerarchia della
    chiave articolo deve comunque saper gestire un campo
    canonico mancante.
    """

    if codice_tipo == TOKEN_NESSUN_CODICE:
        return TOKEN_NESSUN_CODICE

    if campo_canonico is not None and not pd.isna(campo_canonico):
        return campo_canonico

    if codice_tipo is not None and not pd.isna(codice_tipo):
        return f"NON_MAPPATO|{codice_tipo}"

    return TOKEN_NESSUN_CODICE


df_xml_codici["CampoPivot"] = [
    campo_pivot(tipo, campo)
    for tipo, campo in zip(
        df_xml_codici["CodiceTipoXML"],
        df_xml_codici["CampoCanonico"],
    )
]


# --------------------------------------------------------------
# 3. PIVOT
# --------------------------------------------------------------
# Per ogni coppia (linea, campo) si tiene il primo valore non
# nullo nell'ordine di comparsa nell'XML.

codici_pivot = (
    df_xml_codici.loc[
        df_xml_codici["CodiceValoreNormalizzato"].notna()
    ]
    .sort_values(
        [COL_CHIAVE_RIGA_XML, "ProgressivoCodiceXML"],
        na_position="last",
    )
    .drop_duplicates(
        subset=[COL_CHIAVE_RIGA_XML, "CampoPivot"],
        keep="first",
    )
    .loc[
        :,
        [
            COL_CHIAVE_RIGA_XML,
            "CampoPivot",
            "CodiceValoreNormalizzato",
        ],
    ]
)

df_pivot = (
    codici_pivot
    .pivot(
        index=COL_CHIAVE_RIGA_XML,
        columns="CampoPivot",
        values="CodiceValoreNormalizzato",
    )
    .reset_index()
)

df_pivot.columns.name = None

if TOKEN_NESSUN_CODICE in df_pivot.columns:
    df_pivot = df_pivot.drop(columns=[TOKEN_NESSUN_CODICE])

df_pivot = df_pivot.rename(columns=CAMPI_CANONICI_PIVOT)

# Le tre colonne canoniche devono esistere sempre, anche quando
# nessun XML le contiene: la chiave articolo ne ha bisogno.
for colonna_pivot in (
    NOME_CAMPO_EAN,
    NOME_CAMPO_FORNITORE,
    NOME_CAMPO_CLIENTE,
):
    if colonna_pivot not in df_pivot.columns:
        df_pivot[colonna_pivot] = pd.NA

    df_pivot[colonna_pivot] = df_pivot[colonna_pivot].astype("string")


# --------------------------------------------------------------
# 4. DATI DI LINEA
# --------------------------------------------------------------

COLONNE_LINEA = [
    "NumeroFattura",
    "NumeroLinea",
    "TipoCessionePrestazione",
    "Descrizione",
    "Quantita",
    "UnitaMisura",
    "PrezzoUnitario",
    "PrezzoTotale",
    "DataFattura",
    "NumeroDDT",
    "NumeroDDT_Chiave",
    "TipoAbbinamentoDDT",
]

# Campi previsti dal tracciato ma non ancora estratti dalla cella
# [3]: vengono aggregati automaticamente quando esisteranno, senza
# altre modifiche.
COLONNE_LINEA_OPZIONALI = [
    "AliquotaIVA",
    "CodiceFiscale",
    "Denominazione",
    "DataDDT",
]

df_xml_righe = pd.DataFrame(
    {
        COL_CHIAVE_RIGA_XML: df_xml_fatture[COL_CHIAVE_RIGA_XML],
        "IdFile": df_xml_fatture["CEAS_XML"],
        "CFOR": df_xml_fatture[COL_CFOR_XML],
    }
)

for colonna_linea in COLONNE_LINEA:
    df_xml_righe[colonna_linea] = df_xml_fatture[colonna_linea]

for colonna_linea in COLONNE_LINEA_OPZIONALI:
    if colonna_linea in df_xml_fatture.columns:
        df_xml_righe[colonna_linea] = df_xml_fatture[colonna_linea]

mancanti = [
    colonna
    for colonna in COLONNE_LINEA_OPZIONALI
    if colonna not in df_xml_fatture.columns
]

if mancanti:
    print("Campi non ancora estratti:", ", ".join(mancanti))
    print("  → le righe aggregate non li conterranno")

df_xml_righe["NumeroFattura"] = (
    df_xml_righe["NumeroFattura"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

df_xml_righe["TipoCessionePrestazione"] = (
    df_xml_righe["TipoCessionePrestazione"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

df_xml_righe = df_xml_righe.merge(
    df_pivot,
    how="left",
    on=COL_CHIAVE_RIGA_XML,
    validate="one_to_one",
)


# --------------------------------------------------------------
# 5. CHIAVE ARTICOLO XML
# --------------------------------------------------------------
# Gerarchia della chiave:
#   EAN → codice articolo cliente → codice articolo fornitore
#   → natura della riga → riga descrittiva.

def chiave_articolo_xml(
    ean,
    cliente,
    fornitore,
    tipo_cessione,
    numero_linea,
) -> str:

    if ean is not None and not pd.isna(ean):
        return f"EAN|{ean}"

    if cliente is not None and not pd.isna(cliente):
        return f"CART|{cliente}"

    if fornitore is not None and not pd.isna(fornitore):
        return f"FORN|{fornitore}"

    numero = in_testo(numero_linea)

    if tipo_cessione in NATURA_RIGA:
        return f"{NATURA_RIGA[tipo_cessione]}|{numero}"

    if numero is None:
        return "RIGA DESCRITTIVA"

    return f"RIGA DESCRITTIVA|{numero}"


df_xml_righe["ChiaveArticoloXML"] = [
    chiave_articolo_xml(ean, cliente, fornitore, tipo, linea)
    for ean, cliente, fornitore, tipo, linea in zip(
        df_xml_righe[NOME_CAMPO_EAN],
        df_xml_righe[NOME_CAMPO_CLIENTE],
        df_xml_righe[NOME_CAMPO_FORNITORE],
        df_xml_righe["TipoCessionePrestazione"],
        df_xml_righe["NumeroLinea"],
    )
]


# --------------------------------------------------------------
# 6. AGGREGAZIONE
# --------------------------------------------------------------

CHIAVI_AGGREGAZIONE = [
    "IdFile",
    "CFOR",
    "NumeroFattura",
    "NumeroDDT",
    "NumeroDDT_Chiave",
]

if "DataDDT" in df_xml_righe.columns:
    CHIAVI_AGGREGAZIONE.append("DataDDT")

CHIAVI_AGGREGAZIONE.append("ChiaveArticoloXML")

# Primo valore non nullo del gruppo
COLONNE_PRIMO_VALORE = [
    "Descrizione",
    "UnitaMisura",
    NOME_CAMPO_EAN,
    NOME_CAMPO_FORNITORE,
    NOME_CAMPO_CLIENTE,
    "TipoCessionePrestazione",
    "TipoAbbinamentoDDT",
    "DataFattura",
] + [
    colonna
    for colonna in COLONNE_LINEA_OPZIONALI
    if colonna in df_xml_righe.columns
    # DataDDT è già una chiave di aggregazione: ripeterla anche
    # come colonna di output farebbe fallire il groupby.
    and colonna not in CHIAVI_AGGREGAZIONE
]

# Somma, ma nulla se il gruppo non ha nessun valore
COLONNE_SOMMA = ["Quantita", "PrezzoTotale"]


def somma_o_nulla(serie):
    return serie.sum(min_count=1)


def numeri_linea_xml(serie):
    valori = sorted({int(v) for v in serie.dropna()})

    if not valori:
        return pd.NA

    return ", ".join(f"{v:02d}" for v in valori)


specifiche_aggregazione = {
    colonna: (colonna, "first")
    for colonna in COLONNE_PRIMO_VALORE
}

for colonna in COLONNE_SOMMA:
    specifiche_aggregazione[colonna] = (colonna, somma_o_nulla)

specifiche_aggregazione["NumeriLineaXML"] = (
    "NumeroLinea",
    numeri_linea_xml,
)
specifiche_aggregazione["NumeroRigheAggregate"] = ("NumeroLinea", "size")

df_xml_righe = (
    df_xml_righe
    .groupby(CHIAVI_AGGREGAZIONE, dropna=False)
    .agg(**specifiche_aggregazione)
    .reset_index()
)


# --------------------------------------------------------------
# 7. CHIAVE AGGREGATA E INDICE
# --------------------------------------------------------------

df_xml_righe["ChiaveAggregataXML"] = (
    df_xml_righe["CFOR"].astype("string").fillna("NO_CFOR")
    + "|"
    + df_xml_righe["IdFile"].astype("string").fillna("")
    + "|"
    + df_xml_righe["NumeroFattura"].astype("string").fillna("")
    + "|"
    + df_xml_righe["NumeroDDT"].astype("string").fillna("NO_DDT")
    + "|"
    + df_xml_righe["ChiaveArticoloXML"].astype("string")
)

df_xml_righe = df_xml_righe.reset_index(drop=True)

df_xml_righe["Indicerighe"] = pd.Series(
    range(1, len(df_xml_righe) + 1),
    dtype="Int64",
)


# --------------------------------------------------------------
# 8. PREZZO UNITARIO NETTO
# --------------------------------------------------------------

df_xml_righe["PrezzoUnitarioNetto"] = (
    df_xml_righe["PrezzoTotale"] / df_xml_righe["Quantita"]
).where(df_xml_righe["Quantita"].notna() & df_xml_righe["Quantita"].ne(0))


# --------------------------------------------------------------
# 9. RIEPILOGO
# --------------------------------------------------------------

print(f"Righe XML di partenza: {len(df_xml_fatture):,}")
print(f"Righe aggregate:       {len(df_xml_righe):,}")
print(
    "Righe con più di una linea aggregata: "
    f"{int((df_xml_righe['NumeroRigheAggregate'] > 1).sum()):,}"
)

display(
    df_xml_righe["ChiaveArticoloXML"]
    .str.split("|")
    .str[0]
    .value_counts(dropna=False)
    .rename_axis("Origine chiave articolo")
    .reset_index(name="Righe")
)

display(df_xml_righe.head(20))


# %% [14]


# ==============================================================
# ABBINAMENTO DDT  (map_ddt)
# ==============================================================
# Associa i DDT dell'XML ai DDT dei carichi all'interno di ogni
# coppia CFOR + numero fattura, su tre livelli:
#
#   LIVELLO 1  TESTO
#              chiave DDT normalizzata identica sui due lati;
#   LIVELLO 2  DATA_UNIVOCA
#              sulla stessa data resta un solo DDT XML e un solo
#              DDT carico fra quelli non ancora abbinati;
#   LIVELLO 3  UNICO_RESIDUO
#              dopo i livelli precedenti resta un solo DDT per
#              parte. Se una delle due date manca l'abbinamento
#              è comunque accettato, con metodo distinto.
#
# I DDT non abbinati NON ricevono una chiave comune: chi li usa
# costruisce una chiave "orfano" che, per costruzione, non può
# incontrare nulla sull'altro lato.
#
# DataDDT arriva dalla cella [3] ed è valorizzata solo quando la
# linea è collegata a un solo DDT, oppure a più DDT con la stessa
# data. Senza data il livello 2 non può scattare e il livello 3
# usa il metodo UNICO_RESIDUO_DATA_MANCANTE.
# ==============================================================

# Flag di abilitazione dei livelli di abbinamento. Il dizionario
# non li prevede ancora, quindi valgono per tutti i fornitori:
# per disattivarli basta portare a False il valore qui sotto.
ABILITA_MATCH_DATA = True
ABILITA_MATCH_UNICO_RESIDUO = True

CHIAVE_ORFANO_XML = "XML_ORFANO"
CHIAVE_ORFANO_CARICO = "CARICO_ORFANO"

# Valore di OrigineChiaveDDT per le righe in cui l'XML non riporta
# alcun DDT e la chiave viene presa dal DDT unico del carico.
CHIAVE_DDT_DERIVATA = "DDT_UNICO_CARICO"

# Più DDT sul carico per la stessa fattura: nessuna derivazione,
# il dato è ambiguo e va lasciato non abbinato.
CHIAVE_DDT_AMBIGUA = "DDT_AMBIGUO"

PRIORITA_METODO = {
    "TESTO": 1,
    "DATA_UNIVOCA": 2,
    "UNICO_RESIDUO": 3,
    "UNICO_RESIDUO_DATA_MANCANTE": 3,
}

COLONNE_MAP_DDT = [
    "CFOR",
    "NumeroFattura",
    "ChiaveDDT_XML",
    "ChiaveDDT_Carico",
    "ChiaveDDT_Comune",
    "MetodoMatchDDT",
    "NumeroDDT_XML",
    "NDOGIAB_Carico",
    "DataDDT_XML",
    "TDATIAB_Carico",
]


# --------------------------------------------------------------
# FUNZIONI DI SERVIZIO
# --------------------------------------------------------------
# valore_valido e maschera_valida sono usati anche dalla cella
# successiva: considerano valido tutto ciò che non è nullo e non è
# una stringa vuota.

def valore_valido(valore) -> bool:
    """
    Vero se il valore non è nullo e non è una stringa vuota.
    """

    if valore is None:
        return False

    if not isinstance(valore, str) and pd.isna(valore):
        return False

    return str(valore).strip() != ""


def maschera_valida(serie) -> pd.Series:
    """
    Versione vettoriale di valore_valido.
    """

    testo = serie.astype("string").str.strip()

    return testo.notna() & (testo != "")


def primo_valido(serie):
    valori = serie.dropna()

    return valori.iloc[0] if len(valori) else pd.NA


# --------------------------------------------------------------
# 1. TABELLE DI PARTENZA
# --------------------------------------------------------------

COLONNE_XML_DDT = [
    "IdFile",
    "CFOR",
    "NumeroFattura",
    "NumeroDDT",
    "NumeroDDT_Chiave",
]

if "DataDDT" in df_xml_righe.columns:
    COLONNE_XML_DDT = COLONNE_XML_DDT + ["DataDDT"]

xml_ddt = (
    df_xml_righe[COLONNE_XML_DDT]
    .drop_duplicates()
    .copy()
)

if "DataDDT" not in xml_ddt.columns:
    xml_ddt["DataDDT"] = pd.NaT

xml_ddt = xml_ddt.loc[
    xml_ddt["CFOR"].notna()
    & xml_ddt["NumeroFattura"].notna()
    & maschera_valida(xml_ddt["NumeroDDT_Chiave"])
].copy()

carichi_ddt = (
    df_fatture[
        [
            COL_CFOR,
            COL_NDOGIAA,
            COL_NDOGIAB,
            "NDOGIAB_Chiave",
            COL_TDATIAB,
        ]
    ]
    .drop_duplicates()
    .copy()
)

carichi_ddt = carichi_ddt.loc[
    carichi_ddt[COL_CFOR].notna()
    & carichi_ddt[COL_NDOGIAA].notna()
    & maschera_valida(carichi_ddt["NDOGIAB_Chiave"])
].copy()

print(f"DDT distinti lato XML:     {len(xml_ddt):,}")
print(f"DDT distinti lato carichi: {len(carichi_ddt):,}")


# --------------------------------------------------------------
# 2. ABBINAMENTO DI UN GRUPPO CFOR + FATTURA
# --------------------------------------------------------------

def abbina_gruppo(
    df_gruppo_xml,
    df_gruppo_carichi,
    usa_data: bool,
    usa_residuo: bool,
) -> list:
    """
    Restituisce l'elenco delle coppie abbinate del gruppo.
    """

    xml_g = df_gruppo_xml[
        ["NumeroDDT", "NumeroDDT_Chiave", "DataDDT"]
    ].copy()

    car_g = df_gruppo_carichi[
        [COL_NDOGIAB, "NDOGIAB_Chiave", COL_TDATIAB]
    ].copy()

    def crea_coppia(chiave_xml, chiave_carico, metodo) -> dict:

        righe_xml = xml_g.loc[
            xml_g["NumeroDDT_Chiave"] == chiave_xml
        ]

        righe_carico = car_g.loc[
            car_g["NDOGIAB_Chiave"] == chiave_carico
        ]

        return {
            "ChiaveDDT_XML": chiave_xml,
            "ChiaveDDT_Carico": chiave_carico,
            "MetodoMatchDDT": metodo,
            "NumeroDDT_XML": primo_valido(righe_xml["NumeroDDT"]),
            "NDOGIAB_Carico": primo_valido(righe_carico[COL_NDOGIAB]),
            "DataDDT_XML": primo_valido(righe_xml["DataDDT"]),
            "TDATIAB_Carico": primo_valido(righe_carico[COL_TDATIAB]),
        }

    chiavi_xml = list(
        dict.fromkeys(xml_g["NumeroDDT_Chiave"].dropna())
    )

    chiavi_carico = list(
        dict.fromkeys(car_g["NDOGIAB_Chiave"].dropna())
    )

    # ---- LIVELLO 1: TESTO ------------------------------------
    comuni = [
        chiave for chiave in chiavi_xml if chiave in set(chiavi_carico)
    ]

    coppie = [crea_coppia(chiave, chiave, "TESTO") for chiave in comuni]

    residuo_xml = [
        chiave for chiave in chiavi_xml if chiave not in set(comuni)
    ]

    residuo_carico = [
        chiave for chiave in chiavi_carico if chiave not in set(comuni)
    ]

    # ---- LIVELLO 2: DATA_UNIVOCA -----------------------------
    coppie_data = []

    if usa_data:

        xml_per_data = (
            xml_g.loc[
                xml_g["NumeroDDT_Chiave"].isin(residuo_xml),
                ["DataDDT", "NumeroDDT_Chiave"],
            ]
            .drop_duplicates()
            .dropna(subset=["DataDDT"])
            .groupby("DataDDT", dropna=False)
            .agg(
                NumeroDDTXML=("NumeroDDT_Chiave", "size"),
                ChiaveXML=("NumeroDDT_Chiave", "first"),
            )
            .reset_index()
        )

        carichi_per_data = (
            car_g.loc[
                car_g["NDOGIAB_Chiave"].isin(residuo_carico),
                [COL_TDATIAB, "NDOGIAB_Chiave"],
            ]
            .drop_duplicates()
            .dropna(subset=[COL_TDATIAB])
            .groupby(COL_TDATIAB, dropna=False)
            .agg(
                NumeroDDTCarico=("NDOGIAB_Chiave", "size"),
                ChiaveCarico=("NDOGIAB_Chiave", "first"),
            )
            .reset_index()
        )

        if not xml_per_data.empty and not carichi_per_data.empty:

            date_univoche = (
                xml_per_data
                .merge(
                    carichi_per_data,
                    how="inner",
                    left_on="DataDDT",
                    right_on=COL_TDATIAB,
                )
                .loc[
                    lambda df: (df["NumeroDDTXML"] == 1)
                    & (df["NumeroDDTCarico"] == 1)
                    & df["ChiaveXML"].notna()
                    & df["ChiaveCarico"].notna()
                ]
            )

            coppie_data = [
                crea_coppia(
                    riga.ChiaveXML,
                    riga.ChiaveCarico,
                    "DATA_UNIVOCA",
                )
                for riga in date_univoche.itertuples()
            ]

        usate_xml = {coppia["ChiaveDDT_XML"] for coppia in coppie_data}
        usate_carico = {
            coppia["ChiaveDDT_Carico"] for coppia in coppie_data
        }

        residuo_xml = [
            chiave for chiave in residuo_xml if chiave not in usate_xml
        ]

        residuo_carico = [
            chiave
            for chiave in residuo_carico
            if chiave not in usate_carico
        ]

    # ---- LIVELLO 3: UNICO_RESIDUO ----------------------------
    coppie_residuo = []

    if (
        usa_residuo
        and len(residuo_xml) == 1
        and len(residuo_carico) == 1
    ):

        chiave_xml = residuo_xml[0]
        chiave_carico = residuo_carico[0]

        data_xml = primo_valido(
            xml_g.loc[
                xml_g["NumeroDDT_Chiave"] == chiave_xml,
                "DataDDT",
            ]
        )

        data_carico = primo_valido(
            car_g.loc[
                car_g["NDOGIAB_Chiave"] == chiave_carico,
                COL_TDATIAB,
            ]
        )

        data_mancante = (
            data_xml is pd.NaT
            or data_carico is pd.NaT
            or pd.isna(data_xml)
            or pd.isna(data_carico)
        )

        compatibili = data_mancante or data_xml == data_carico

        if compatibili:

            metodo = (
                "UNICO_RESIDUO_DATA_MANCANTE"
                if data_mancante
                else "UNICO_RESIDUO"
            )

            coppie_residuo = [
                crea_coppia(chiave_xml, chiave_carico, metodo)
            ]

    return coppie + coppie_data + coppie_residuo


# --------------------------------------------------------------
# 3. ABBINAMENTO DI TUTTI I GRUPPI
# --------------------------------------------------------------

gruppi_xml = {
    chiave: gruppo[["NumeroDDT", "NumeroDDT_Chiave", "DataDDT"]]
    for chiave, gruppo in xml_ddt.groupby(
        ["CFOR", "NumeroFattura"], dropna=False
    )
}

gruppi_carichi = {
    chiave: gruppo[[COL_NDOGIAB, "NDOGIAB_Chiave", COL_TDATIAB]]
    for chiave, gruppo in carichi_ddt.groupby(
        [COL_CFOR, COL_NDOGIAA], dropna=False
    )
}

record_coppie = []

for chiave_gruppo, gruppo_xml in gruppi_xml.items():

    gruppo_carichi = gruppi_carichi.get(chiave_gruppo)

    # Inner join: senza il carico non c'è nulla da abbinare
    if gruppo_carichi is None:
        continue

    for coppia in abbina_gruppo(
        gruppo_xml,
        gruppo_carichi,
        ABILITA_MATCH_DATA,
        ABILITA_MATCH_UNICO_RESIDUO,
    ):

        coppia = dict(coppia)
        coppia["CFOR"] = chiave_gruppo[0]
        coppia["NumeroFattura"] = chiave_gruppo[1]
        record_coppie.append(coppia)


# --------------------------------------------------------------
# 4. PRIORITÀ E CHIAVE COMUNE
# --------------------------------------------------------------

if record_coppie:

    map_ddt = pd.DataFrame(record_coppie)

    map_ddt = map_ddt.loc[
        map_ddt["ChiaveDDT_XML"].notna()
        & map_ddt["ChiaveDDT_Carico"].notna()
    ].copy()

    map_ddt["PrioritaMetodo"] = (
        map_ddt["MetodoMatchDDT"]
        .map(PRIORITA_METODO)
        .astype("Int64")
    )

    # Uno stesso DDT XML può comparire con metodi diversi:
    # vince il livello più sicuro.
    map_ddt = (
        map_ddt
        .sort_values(
            [
                "CFOR",
                "NumeroFattura",
                "ChiaveDDT_XML",
                "PrioritaMetodo",
            ],
            na_position="last",
        )
        .drop_duplicates(
            subset=["CFOR", "NumeroFattura", "ChiaveDDT_XML"],
            keep="first",
        )
        .sort_values(
            [
                "CFOR",
                "NumeroFattura",
                "ChiaveDDT_Carico",
                "PrioritaMetodo",
            ],
            na_position="last",
        )
        .drop_duplicates(
            subset=["CFOR", "NumeroFattura", "ChiaveDDT_Carico"],
            keep="first",
        )
        .reset_index(drop=True)
    )

    # La colonna deve restare di tipo data anche quando è tutta
    # vuota, altrimenti l'export la tratterebbe come testo.
    map_ddt["DataDDT_XML"] = pd.to_datetime(
        map_ddt["DataDDT_XML"],
        errors="coerce",
    )

    # Il lato carico è il riferimento canonico della chiave
    map_ddt["ChiaveDDT_Comune"] = (
        map_ddt["CFOR"].astype("string")
        + "|"
        + map_ddt["NumeroFattura"].astype("string")
        + "|"
        + map_ddt["ChiaveDDT_Carico"].astype("string")
    )

    map_ddt = map_ddt[COLONNE_MAP_DDT]

else:

    map_ddt = pd.DataFrame(
        {
            colonna: pd.Series(dtype="string")
            for colonna in COLONNE_MAP_DDT
            if colonna != "DataDDT_XML"
        }
    )

    map_ddt["DataDDT_XML"] = pd.Series(dtype="datetime64[ns]")

    map_ddt = map_ddt[COLONNE_MAP_DDT]

    print("Nessun abbinamento DDT: map_ddt è vuota")


# --------------------------------------------------------------
# 5. RIEPILOGO
# --------------------------------------------------------------

print(f"Coppie abbinate: {len(map_ddt):,}")

if len(map_ddt) > 0:

    display(
        map_ddt["MetodoMatchDDT"]
        .value_counts(dropna=False)
        .rename_axis("MetodoMatchDDT")
        .reset_index(name="Coppie")
    )

    display(map_ddt.head(20))


# %% [15]


# ==============================================================
# MERGE FRA XML E CARICHI
# ==============================================================
# Esegue il merge fra i due lati del processo.
#
#   1. ogni riga XML riceve la chiave DDT comune (map_ddt);
#   2. ogni riga carico riceve la stessa chiave;
#   3. tre lookup in cascata, in ordine di priorità:
#          EAN  →  CART  →  CALA
#      ciascuno restituisce l'Indice più basso fra le righe
#      carico compatibili;
#   4. full outer fra lato XML e lato carichi su quell'Indice;
#   5. StatoDocumento, StatoAbbinamento, NumeroDDT_Comune.
#
# Una riga XML che non trova nulla resta "Solo XML"; una riga
# carico non raggiunta da nessuna riga XML resta "Solo carico".
# ==============================================================


# --------------------------------------------------------------
# 1. CHIAVE DDT COMUNE SUL LATO XML
# --------------------------------------------------------------

df_xml_merge = df_xml_righe.copy()

df_xml_merge = df_xml_merge.merge(
    map_ddt[
        [
            "CFOR",
            "NumeroFattura",
            "ChiaveDDT_XML",
            "ChiaveDDT_Comune",
            "MetodoMatchDDT",
        ]
    ],
    how="left",
    left_on=["CFOR", "NumeroFattura", "NumeroDDT_Chiave"],
    right_on=["CFOR", "NumeroFattura", "ChiaveDDT_XML"],
    validate="many_to_one",
)

df_xml_merge = df_xml_merge.drop(columns=["ChiaveDDT_XML"])

df_xml_merge = df_xml_merge.rename(
    columns={"ChiaveDDT_Comune": "ChiaveDDT_ComuneDaCoppia"}
)

# Chiave "orfano": non può incontrare nulla sul lato carichi
orfan_xml = (
    df_xml_merge["CFOR"].astype("string")
    + "|"
    + df_xml_merge["NumeroFattura"].astype("string")
    + "|"
    + CHIAVE_ORFANO_XML
    + "|"
    + df_xml_merge["NumeroDDT_Chiave"].astype("string")
)

valide_xml = maschera_valida(df_xml_merge["NumeroDDT_Chiave"])

df_xml_merge["ChiaveDDT_ComuneXML"] = pd.Series(
    pd.NA,
    index=df_xml_merge.index,
    dtype="string",
)

df_xml_merge.loc[valide_xml, "ChiaveDDT_ComuneXML"] = (
    df_xml_merge.loc[valide_xml, "ChiaveDDT_ComuneDaCoppia"]
    .fillna(orfan_xml.loc[valide_xml])
    .astype("string")
)

df_xml_merge = df_xml_merge.reset_index(drop=True)

df_xml_merge["ID_Riga_XML"] = pd.Series(
    range(1, len(df_xml_merge) + 1),
    dtype="Int64",
)


# --------------------------------------------------------------
# 2. CHIAVE DDT COMUNE SUL LATO CARICHI
# --------------------------------------------------------------

df_carichi_merge = df_fatture.copy()

df_carichi_merge = df_carichi_merge.merge(
    map_ddt[
        [
            "CFOR",
            "NumeroFattura",
            "ChiaveDDT_Carico",
            "ChiaveDDT_Comune",
            "NumeroDDT_XML",
        ]
    ],
    how="left",
    left_on=[COL_CFOR, COL_NDOGIAA, "NDOGIAB_Chiave"],
    right_on=["CFOR", "NumeroFattura", "ChiaveDDT_Carico"],
    validate="many_to_one",
)

df_carichi_merge = df_carichi_merge.drop(
    columns=["CFOR", "NumeroFattura", "ChiaveDDT_Carico"]
)

df_carichi_merge = df_carichi_merge.rename(
    columns={
        "ChiaveDDT_Comune": "ChiaveDDT_ComuneDaCoppia",
        "NumeroDDT_XML": "NumeroDDT_XML_Abbinato",
    }
)

orfan_carico = (
    df_carichi_merge[COL_CFOR].astype("string")
    + "|"
    + df_carichi_merge[COL_NDOGIAA].astype("string")
    + "|"
    + CHIAVE_ORFANO_CARICO
    + "|"
    + df_carichi_merge["NDOGIAB_Chiave"].astype("string")
)

valide_carico = maschera_valida(df_carichi_merge["NDOGIAB_Chiave"])

df_carichi_merge["ChiaveDDT_ComuneCarico"] = pd.Series(
    pd.NA,
    index=df_carichi_merge.index,
    dtype="string",
)

df_carichi_merge.loc[valide_carico, "ChiaveDDT_ComuneCarico"] = (
    df_carichi_merge.loc[valide_carico, "ChiaveDDT_ComuneDaCoppia"]
    .fillna(orfan_carico.loc[valide_carico])
    .astype("string")
)

# --------------------------------------------------------------
# 2-bis. XML SENZA DDT: CHIAVE DERIVATA DAL CARICO
# --------------------------------------------------------------
# L'XML può non riportare alcun DatiDDT (fattura senza bolla
# indicata) mentre il carico la riporta sempre. In quel caso la
# riga XML non ha chiave e non potrebbe agganciarsi a nulla, pur
# avendo il documento presente nei carichi.
#
# Regola: se per la stessa coppia CFOR + fattura il lato carichi
# ha UN SOLO DDT, si usa quello. Con più DDT il dato è ambiguo e
# non si deriva nulla: meglio non abbinare che abbinare a caso.
#
# La chiave derivata è letta direttamente dal lato carichi
# (ChiaveDDT_ComuneCarico): così coincide per costruzione con
# quella su cui lavorano le lookup, orfano compreso.

COL_CHIAVE_CARICO = "ChiaveDDT_ComuneCarico"

ddt_per_fattura = (
    df_carichi_merge.loc[
        maschera_valida(df_carichi_merge[COL_CHIAVE_CARICO]),
        [COL_CFOR, COL_NDOGIAA, COL_NDOGIAB, COL_CHIAVE_CARICO],
    ]
    .drop_duplicates()
)

ddt_distinti = (
    ddt_per_fattura
    .groupby([COL_CFOR, COL_NDOGIAA])[COL_CHIAVE_CARICO]
    .nunique()
    .rename("DDTDistinti")
    .reset_index()
)

ddt_unico_carico = (
    ddt_per_fattura
    .merge(ddt_distinti, how="inner", on=[COL_CFOR, COL_NDOGIAA])
    .loc[lambda df: df["DDTDistinti"] == 1]
    .rename(
        columns={
            COL_CFOR: "CFOR",
            COL_NDOGIAA: "NumeroFattura",
            COL_NDOGIAB: "NDOGIAB_Derivato",
            COL_CHIAVE_CARICO: "ChiaveDDT_ComuneDerivata",
        }
    )
    .loc[
        :,
        ["CFOR", "NumeroFattura", "NDOGIAB_Derivato", "ChiaveDDT_ComuneDerivata"],
    ]
)

# Quanti DDT ha il carico per quella fattura: serve sia a
# derivare (uno solo) sia a distinguere il caso ambiguo (più di
# uno) da quello in cui il carico non ha alcun DDT.
df_xml_merge = df_xml_merge.merge(
    ddt_distinti.rename(
        columns={
            COL_CFOR: "CFOR",
            COL_NDOGIAA: "NumeroFattura",
            "DDTDistinti": "DDTDistintiCarico",
        }
    ),
    how="left",
    on=["CFOR", "NumeroFattura"],
    validate="many_to_one",
)

df_xml_merge = df_xml_merge.merge(
    ddt_unico_carico,
    how="left",
    on=["CFOR", "NumeroFattura"],
    validate="many_to_one",
)

da_derivare = (
    ~maschera_valida(df_xml_merge["ChiaveDDT_ComuneXML"])
    & maschera_valida(df_xml_merge["ChiaveDDT_ComuneDerivata"])
)

df_xml_merge.loc[da_derivare, "ChiaveDDT_ComuneXML"] = (
    df_xml_merge.loc[da_derivare, "ChiaveDDT_ComuneDerivata"]
    .astype("string")
)

print(
    "Righe XML senza DDT agganciate al DDT unico del carico: "
    f"{int(da_derivare.sum()):,}"
)


# --------------------------------------------------------------
# 2-ter. ORIGINE DELLA CHIAVE
# --------------------------------------------------------------
# Serve a distinguere un abbinamento vero da uno derivato, così un
# DDT ricostruito resta riconoscibile nel risultato.

origine_chiave = pd.Series(
    "NESSUNA_CHIAVE",
    index=df_xml_merge.index,
    dtype="string",
)

origine_chiave[df_xml_merge["ChiaveDDT_ComuneDaCoppia"].notna()] = "MAPPATO"

origine_chiave[
    origine_chiave.eq("NESSUNA_CHIAVE")
    & maschera_valida(df_xml_merge["NumeroDDT_Chiave"])
] = "ORFANO"

origine_chiave[
    origine_chiave.eq("NESSUNA_CHIAVE")
    & maschera_valida(df_xml_merge["ChiaveDDT_ComuneDerivata"])
] = CHIAVE_DDT_DERIVATA

# Più DDT sul carico e nessun DDT nell'XML: si segnala, non si
# sceglie. Lo stato dedicato rende il caso leggibile in output.
origine_chiave[
    origine_chiave.eq("NESSUNA_CHIAVE")
    & df_xml_merge["DDTDistintiCarico"].gt(1).fillna(False)
] = CHIAVE_DDT_AMBIGUA

df_xml_merge["OrigineChiaveDDT"] = origine_chiave

print(
    "Righe XML con più DDT sul carico (ambigue, non abbinate): "
    f"{int(origine_chiave.eq(CHIAVE_DDT_AMBIGUA).sum()):,}"
)

df_xml_merge = df_xml_merge.reset_index(drop=True)


# --------------------------------------------------------------
# 3. TABELLE DI LOOKUP
# --------------------------------------------------------------

def crea_lookup_codice(
    df_sorgente,
    colonna_codice,
    nome_id,
    nome_conteggio,
):
    """
    Raggruppa i carichi per fornitore, fattura, DDT comune e
    codice, restituendo l'Indice più basso e il numero di righe.
    """

    valide = (
        maschera_valida(df_sorgente[COL_CFOR])
        & maschera_valida(df_sorgente[COL_NDOGIAA])
        & maschera_valida(df_sorgente["ChiaveDDT_ComuneCarico"])
        & maschera_valida(df_sorgente[colonna_codice])
        & df_sorgente["Indice"].notna()
    )

    return (
        df_sorgente.loc[valide]
        .groupby(
            [
                COL_CFOR,
                COL_NDOGIAA,
                "ChiaveDDT_ComuneCarico",
                colonna_codice,
            ],
            dropna=False,
        )
        .agg(
            **{
                nome_id: ("Indice", "min"),
                nome_conteggio: ("Indice", "size"),
            }
        )
        .reset_index()
    )


# Le lookup lavorano sulle chiavi normalizzate: il lato XML porta
# valori già normalizzati dal dizionario (cella [12]), il lato
# carichi dalle chiavi costruite nella cella [10].
lookup_ean = crea_lookup_codice(
    df_carichi_merge, "CEAN_Chiave", "ID_Riga_Ordine_EAN", "NumeroMatchEAN"
)

lookup_cart = crea_lookup_codice(
    df_carichi_merge, "CART_Chiave", "ID_Riga_Ordine_CART", "NumeroMatchCART"
)

lookup_cala = crea_lookup_codice(
    df_carichi_merge, "CALA_Chiave", "ID_Riga_Ordine_CALA", "NumeroMatchCALA"
)


# Documento: tutte le righe carico di quel DDT
valide_documento = (
    maschera_valida(df_carichi_merge[COL_CFOR])
    & maschera_valida(df_carichi_merge[COL_NDOGIAA])
    & maschera_valida(df_carichi_merge["ChiaveDDT_ComuneCarico"])
    & df_carichi_merge["Indice"].notna()
)

lookup_documento = (
    df_carichi_merge.loc[valide_documento]
    .groupby(
        [COL_CFOR, COL_NDOGIAA, "ChiaveDDT_ComuneCarico"],
        dropna=False,
    )
    .agg(NumeroRigheDocumento=("Indice", "size"))
    .reset_index()
)


# Fattura: tutte le righe carico di quella fattura
valide_fattura = (
    maschera_valida(df_carichi_merge[COL_CFOR])
    & maschera_valida(df_carichi_merge[COL_NDOGIAA])
    & df_carichi_merge["Indice"].notna()
)

lookup_fattura = (
    df_carichi_merge.loc[valide_fattura]
    .groupby([COL_CFOR, COL_NDOGIAA], dropna=False)
    .agg(
        NumeroRigheFattura=("Indice", "size"),
        NumeroDDTDistintiCarico=("NDOGIAB_Chiave", "nunique"),
        DDTCaricoDisponibili=(
            "NDOGIAB_Chiave",
            lambda serie: ", ".join(
                sorted({str(v) for v in serie.dropna()})
            ),
        ),
        DDTCaricoChiavi=(
            "NDOGIAB_Chiave",
            lambda serie: ", ".join(
                sorted({str(v) for v in serie.dropna()})
            ),
        ),
    )
    .reset_index()
)


# Fattura globale: prescinde dal fornitore
valide_fattura_globale = (
    maschera_valida(df_carichi_merge[COL_NDOGIAA])
    & df_carichi_merge["Indice"].notna()
)

lookup_fattura_globale = (
    df_carichi_merge.loc[valide_fattura_globale]
    .groupby([COL_NDOGIAA], dropna=False)
    .agg(NumeroRigheFatturaGlobale=("Indice", "size"))
    .reset_index()
)


# --------------------------------------------------------------
# 4. LOOKUP EAN → CART → CALA IN CASCATA
# --------------------------------------------------------------

for lookup, nome_id, nome_conteggio, colonna_xml, colonna_carico in (
    (
        lookup_ean,
        "ID_Riga_Ordine_EAN",
        "NumeroMatchEAN",
        NOME_CAMPO_EAN,
        "CEAN_Chiave",
    ),
    (
        lookup_cart,
        "ID_Riga_Ordine_CART",
        "NumeroMatchCART",
        NOME_CAMPO_CLIENTE,
        "CART_Chiave",
    ),
    (
        lookup_cala,
        "ID_Riga_Ordine_CALA",
        "NumeroMatchCALA",
        NOME_CAMPO_FORNITORE,
        "CALA_Chiave",
    ),
):

    df_xml_merge = df_xml_merge.merge(
        lookup,
        how="left",
        left_on=[
            "CFOR",
            "NumeroFattura",
            "ChiaveDDT_ComuneXML",
            colonna_xml,
        ],
        right_on=[
            COL_CFOR,
            COL_NDOGIAA,
            "ChiaveDDT_ComuneCarico",
            colonna_carico,
        ],
        validate="many_to_one",
    )

    df_xml_merge = df_xml_merge.drop(
        columns=[
            COL_CFOR,
            COL_NDOGIAA,
            "ChiaveDDT_ComuneCarico",
            colonna_carico,
        ]
    )


def tipo_match(ean, cart, cala) -> str:

    if ean is not None and not pd.isna(ean):
        return "EAN"

    if cart is not None and not pd.isna(cart):
        return "CART"

    if cala is not None and not pd.isna(cala):
        return "CALA"

    return "Nessun match"


df_xml_merge["TipoMatch"] = [
    tipo_match(ean, cart, cala)
    for ean, cart, cala in zip(
        df_xml_merge["ID_Riga_Ordine_EAN"],
        df_xml_merge["ID_Riga_Ordine_CART"],
        df_xml_merge["ID_Riga_Ordine_CALA"],
    )
]

df_xml_merge["ID_Riga_Ordine"] = (
    df_xml_merge["ID_Riga_Ordine_EAN"]
    .fillna(df_xml_merge["ID_Riga_Ordine_CART"])
    .fillna(df_xml_merge["ID_Riga_Ordine_CALA"])
    .astype("Int64")
)

# I contatori dei livelli inferiori restano vuoti: evita di
# leggere un numero fuori contesto.
df_xml_merge.loc[
    df_xml_merge["TipoMatch"] == "EAN",
    ["NumeroMatchCART", "NumeroMatchCALA"],
] = pd.NA

df_xml_merge.loc[
    df_xml_merge["TipoMatch"] == "CART",
    ["NumeroMatchCALA"],
] = pd.NA


# --------------------------------------------------------------
# 5. STATO DEL DOCUMENTO
# --------------------------------------------------------------

df_xml_merge = df_xml_merge.merge(
    lookup_documento,
    how="left",
    left_on=["CFOR", "NumeroFattura", "ChiaveDDT_ComuneXML"],
    right_on=[COL_CFOR, COL_NDOGIAA, "ChiaveDDT_ComuneCarico"],
    validate="many_to_one",
).drop(
    columns=[COL_CFOR, COL_NDOGIAA, "ChiaveDDT_ComuneCarico"]
)

df_xml_merge = df_xml_merge.merge(
    lookup_fattura,
    how="left",
    left_on=["CFOR", "NumeroFattura"],
    right_on=[COL_CFOR, COL_NDOGIAA],
    validate="many_to_one",
).drop(columns=[COL_CFOR, COL_NDOGIAA])

df_xml_merge = df_xml_merge.merge(
    lookup_fattura_globale,
    how="left",
    left_on=["NumeroFattura"],
    right_on=[COL_NDOGIAA],
    validate="many_to_one",
).drop(columns=[COL_NDOGIAA])


def stato_documento(
    chiave_ddt,
    righe_doc,
    righe_fat,
    righe_glob,
    ddt_distinti_carico=None,
) -> str:
    """
    Stato del documento.

    L'ordine dei controlli conta: il documento trovato viene prima
    della chiave nulla, altrimenti le righe XML senza DDT ma
    agganciate al DDT unico del carico risulterebbero "DDT XML
    nullo" pur essendo state riconciliate. Il dettaglio si legge in
    OrigineChiaveDDT.
    """

    if righe_doc is not None and not pd.isna(righe_doc):
        return "Documento trovato"

    if not valore_valido(chiave_ddt):

        # XML senza DDT ma carico con più bolle: non si può
        # derivare la chiave, il caso resta dichiaratamente
        # ambiguo invece di confondersi con un XML vuoto.
        if (
            ddt_distinti_carico is not None
            and not pd.isna(ddt_distinti_carico)
            and ddt_distinti_carico > 1
        ):
            return "DDT ambiguo"

        return "DDT XML nullo"

    if righe_fat is not None and not pd.isna(righe_fat):
        return "DDT non corrispondente"

    if righe_glob is not None and not pd.isna(righe_glob):
        return "CFOR non corrispondente"

    return "Fattura non trovata"


df_xml_merge["StatoDocumento"] = [
    stato_documento(chiave, documento, fattura, globale, distinti)
    for chiave, documento, fattura, globale, distinti in zip(
        df_xml_merge["NumeroDDT_Chiave"],
        df_xml_merge["NumeroRigheDocumento"],
        df_xml_merge["NumeroRigheFattura"],
        df_xml_merge["NumeroRigheFatturaGlobale"],
        df_xml_merge["DDTDistintiCarico"],
    )
]


# --------------------------------------------------------------
# 6. FULL OUTER CON I CARICHI
# --------------------------------------------------------------

# NumeroDDT originale dell'XML associato da map_ddt: viaggia
# con la riga carico così attraversa il full outer.
ddt_xml_per_indice = (
    df_carichi_merge[["Indice", "NumeroDDT_XML_Abbinato"]]
    .drop_duplicates(subset=["Indice"])
    .copy()
)

carichi_univoci = (
    df_fatture.drop_duplicates(subset=["Indice"])
    .merge(
        ddt_xml_per_indice,
        how="left",
        on="Indice",
        validate="one_to_one",
    )
)

assert carichi_univoci["Indice"].notna().all(), (
    "Esistono righe carico senza Indice: il full outer ne creerebbe"
    " di fantasma"
)

righe_xml = len(df_xml_merge)

df_merge = df_xml_merge.merge(
    carichi_univoci,
    how="outer",
    left_on="ID_Riga_Ordine",
    right_on="Indice",
    validate="many_to_one",
)

df_merge["TipoMatch"] = df_merge["TipoMatch"].fillna("Solo carico")
df_merge["StatoDocumento"] = df_merge["StatoDocumento"].fillna(
    "Solo carico"
)


def stato_abbinamento(id_riga_xml, indice, id_riga_ordine) -> str:

    xml_assente = id_riga_xml is pd.NA or pd.isna(id_riga_xml)
    carico_assente = indice is pd.NA or pd.isna(indice)
    ordine_assente = id_riga_ordine is pd.NA or pd.isna(id_riga_ordine)

    if xml_assente and not carico_assente:
        return "Solo carico"

    if not xml_assente and ordine_assente:
        return "Solo XML"

    if not xml_assente and not carico_assente:
        return "Abbinato"

    return "Da verificare"


df_merge["StatoAbbinamento"] = [
    stato_abbinamento(riga_xml, indice, ordine)
    for riga_xml, indice, ordine in zip(
        df_merge["ID_Riga_XML"],
        df_merge["Indice"],
        df_merge["ID_Riga_Ordine"],
    )
]


# --------------------------------------------------------------
# 7. ANAGRAFICA DEL CODICE FORNITORE
# --------------------------------------------------------------
# Qui si aggancia l'anagrafica per CARF, portandosi dietro CART e
# il numero di CART distinti: segnala i codici fornitore usati su
# più articoli.
#
# La colonna non si chiama più CARF: la cella [9] l'ha rinominata
# in COL_CARF per portarla dentro df_fatture. Si usa quindi
# COL_CARF, non il nome originale.
#
# Nota: df_carf è già stata deduplicata per CART + CFOR con il
# TDEC più recente, quindi il conteggio dei CART distinti lavora
# sull'anagrafica già pulita.

if "df_carf" in globals() and COL_CARF in df_carf.columns:

    anagrafica_valida = df_carf.loc[
        maschera_valida(df_carf[COL_CARF])
    ]

    anagrafica_univoca = (
        anagrafica_valida
        .groupby(COL_CARF, dropna=False)
        .agg(
            CARTDaAnagrafica=("CART", "first"),
            NumeroCARTAnagrafica=("CART", "nunique"),
        )
        .reset_index()
    )

    # Il nome della colonna è esplicito perché df_merge contiene
    # già diverse colonne di codice articolo.
    df_merge = df_merge.merge(
        anagrafica_univoca,
        how="left",
        left_on=NOME_CAMPO_FORNITORE,
        right_on=COL_CARF,
        validate="many_to_one",
    ).drop(columns=[COL_CARF])


# --------------------------------------------------------------
# 8. NUMERO DDT COMUNE
# --------------------------------------------------------------
# Etichetta di matrice, usata per leggere il risultato.
# La regola CANCELLA_DDT ("DDT aggregati") non è prevista.

def numero_ddt_comune(
    numero_ddt_xml,
    numero_ddt_abbinato,
    numero_ddt_derivato,
    ndogiab,
) -> str:

    if valore_valido(numero_ddt_xml):
        return numero_ddt_xml

    if valore_valido(numero_ddt_abbinato):
        return numero_ddt_abbinato

    if valore_valido(numero_ddt_derivato):
        return numero_ddt_derivato

    if valore_valido(ndogiab):
        return ndogiab

    return "Senza DDT"


df_merge["NumeroDDT_Comune"] = [
    numero_ddt_comune(xml, abbinato, derivato, carico)
    for xml, abbinato, derivato, carico in zip(
        df_merge["NumeroDDT"],
        df_merge["NumeroDDT_XML_Abbinato"],
        df_merge["NDOGIAB_Derivato"],
        df_merge[COL_NDOGIAB],
    )
]


# --------------------------------------------------------------
# 8-bis. STATO DEL DDT, A LIVELLO DI BOLLA
# --------------------------------------------------------------
# Il DDT si abbina fra documenti, non fra righe: tutte le righe
# della stessa bolla devono riportare lo stesso stato, anche
# quando la riga articolo non trova il suo riscontro (una riga
# "Solo carico" di una bolla abbinata resta un problema di
# articolo, non di documento).
#
# Per questo lo stato non si ricava dalla singola riga ma dalla
# chiave comune della bolla: si costruisce una tabella
# chiave → stato e la si applica a tutte le righe.
#
# Le etichette sono in italiano perché la colonna finisce nel
# file finale ed è letta da chi riconcilia, non solo da chi
# sviluppa.

ETICHETTE_METODO = {
    "TESTO": "Bolla abbinata per numero",
    "DATA_UNIVOCA": "Bolla abbinata per data",
    "UNICO_RESIDUO": "Bolla abbinata per esclusione",
    "UNICO_RESIDUO_DATA_MANCANTE": (
        "Bolla abbinata per esclusione, data mancante"
    ),
}

STATO_BOLLA_DEDOTTA = "Bolla dedotta dal carico"
STATO_BOLLA_AMBIGUA = "Bolla ambigua: più bolle nel carico"
STATO_BOLLA_XML_ORFANA = "Bolla XML senza riscontro nei carichi"
STATO_BOLLA_CARICO_ORFANA = "Bolla carico senza riscontro nell'XML"
STATO_BOLLA_ASSENTE = "Nessuna bolla"

stato_per_chiave = {}

# 1. Coppie trovate dall'abbinamento: il metodo dice come.
#    Ha la precedenza su tutto, perché è l'unico abbinamento
#    verificato sui dati.
for chiave, metodo in zip(
    map_ddt["ChiaveDDT_Comune"],
    map_ddt["MetodoMatchDDT"],
):
    stato_per_chiave.setdefault(
        chiave,
        ETICHETTE_METODO.get(metodo, metodo),
    )

# 2. Bolle ricostruite dalla derivazione (sezione 2-bis): la loro
#    chiave di norma non compare in map_ddt, perché nell'XML non
#    c'è DDT. Se la stessa bolla risulta abbinata per altre righe,
#    vince l'abbinamento: la derivazione è solo un ripiego.
#    Si prendono solo le righe in cui la derivazione è stata
#    applicata davvero.
chiavi_derivate = set(
    df_xml_merge.loc[
        df_xml_merge["OrigineChiaveDDT"].eq(CHIAVE_DDT_DERIVATA),
        "ChiaveDDT_ComuneXML",
    ].dropna()
)

for chiave in chiavi_derivate:
    stato_per_chiave.setdefault(chiave, STATO_BOLLA_DEDOTTA)

# 3. Bolle XML rimaste senza riscontro, poi quelle dei carichi.
#    Tutto ciò che non è stato abbinato né derivato è, per
#    costruzione, una chiave "orfano" del proprio lato.
for chiave in df_xml_merge["ChiaveDDT_ComuneXML"].dropna().unique():
    stato_per_chiave.setdefault(chiave, STATO_BOLLA_XML_ORFANA)

for chiave in df_carichi_merge["ChiaveDDT_ComuneCarico"].dropna().unique():
    stato_per_chiave.setdefault(chiave, STATO_BOLLA_CARICO_ORFANA)


# La chiave del lato carichi non attraversa il full outer: si
# riprende dall'Indice, che è la chiave del join.
chiave_carico_per_indice = (
    df_carichi_merge[["Indice", "ChiaveDDT_ComuneCarico"]]
    .drop_duplicates(subset=["Indice"])
    .set_index("Indice")["ChiaveDDT_ComuneCarico"]
)

chiave_ddt_carico = df_merge["Indice"].map(chiave_carico_per_indice)


def stato_ddt_riga(chiave_xml, chiave_carico, origine) -> str:
    """
    Stato del DDT per una riga del merge.

    Conta la bolla a cui la riga appartiene, non il fatto che la
    riga sia stata abbinata: una riga "Solo carico" su una bolla
    riconciliata riporta lo stato della bolla.
    """

    # Caso di fattura: l'XML non ha DDT e il carico ne ha più di
    # uno. Non esiste una bolla a cui agganciare la riga.
    if valore_valido(origine) and origine == CHIAVE_DDT_AMBIGUA:
        return STATO_BOLLA_AMBIGUA

    if valore_valido(chiave_xml):
        return stato_per_chiave.get(chiave_xml, STATO_BOLLA_ASSENTE)

    if valore_valido(chiave_carico):
        return stato_per_chiave.get(chiave_carico, STATO_BOLLA_ASSENTE)

    return STATO_BOLLA_ASSENTE


df_merge["StatoDDT"] = [
    stato_ddt_riga(chiave_xml, chiave_carico, origine)
    for chiave_xml, chiave_carico, origine in zip(
        df_merge["ChiaveDDT_ComuneXML"],
        chiave_ddt_carico,
        df_merge["OrigineChiaveDDT"],
    )
]


# --------------------------------------------------------------
# 8-ter. TIPOLOGIA DEL DOCUMENTO
# --------------------------------------------------------------
# CDOCIAA distingue la fattura (60I) dalla nota di credito (60C).
# È un attributo del documento, non della riga: si ricava dai
# carichi, dove c'è, e si riporta anche sulle righe che esistono
# solo nell'XML.
#
# Il numero fattura identifica il documento: la mappa si costruisce
# per numero e non per CFOR + numero, così resta utilizzabile anche
# sulle righe XML senza fornitore (fattura assente dai carichi).
# Un numero che risultasse associato a più tipologie viene lasciato
# fuori dalla mappa: meglio vuoto che un valore scelto a caso.

tipologie = (
    df_fatture[[COL_NDOGIAA, COL_CDOCIAA]]
    .dropna()
    .drop_duplicates()
)

tipologie_multiple = (
    tipologie
    .groupby(COL_NDOGIAA)[COL_CDOCIAA]
    .nunique()
    .loc[lambda serie: serie > 1]
)

tipologia_per_fattura = (
    tipologie
    .loc[lambda df: ~df[COL_NDOGIAA].isin(tipologie_multiple.index)]
    .drop_duplicates(subset=[COL_NDOGIAA])
    .set_index(COL_NDOGIAA)[COL_CDOCIAA]
)

print(
    "Fatture con più di una tipologia documento (escluse dalla"
    f" mappa): {len(tipologie_multiple):,}"
)

df_merge["TipologiaDocumento"] = pd.Series(
    pd.NA,
    index=df_merge.index,
    dtype="string",
)

# 1. dalla riga di carico, quando c'è
df_merge.loc[
    maschera_valida(df_merge[COL_CDOCIAA]),
    "TipologiaDocumento",
] = df_merge.loc[
    maschera_valida(df_merge[COL_CDOCIAA]),
    COL_CDOCIAA,
].astype("string")

# 2. per le righe Solo XML, dal carico della stessa fattura.
#    Resta vuoto quando la fattura non è nei carichi: oggi non c'è
#    un'altra fonte, il tipo SDI dell'XML (TD01, TD04, ...) è un
#    codice diverso e non è mappato su CDOCIAA.
da_carico = (
    ~maschera_valida(df_merge["TipologiaDocumento"])
    & maschera_valida(df_merge["NumeroFattura"])
)

df_merge.loc[da_carico, "TipologiaDocumento"] = (
    df_merge.loc[da_carico, "NumeroFattura"]
    .map(tipologia_per_fattura)
    .astype("string")
)

print(
    "Righe con tipologia documento ricavata dal carico: "
    f"{int((da_carico & maschera_valida(df_merge['TipologiaDocumento'])).sum()):,}"
)
print(
    "Righe senza tipologia documento (fattura non presente nei"
    f" carichi): {int(df_merge['TipologiaDocumento'].isna().sum()):,}"
)


# --------------------------------------------------------------
# 9. RIEPILOGO
# --------------------------------------------------------------

print(f"Righe XML:      {righe_xml:,}")
print(f"Righe carichi:  {len(carichi_univoci):,}")
print(f"Righe merge:    {len(df_merge):,}")

for colonna_stato in (
    "StatoAbbinamento",
    "TipoMatch",
    "StatoDocumento",
    "MetodoMatchDDT",
    "StatoDDT",
):

    print()
    print(colonna_stato)

    display(
        df_merge[colonna_stato]
        .value_counts(dropna=False)
        .rename_axis(colonna_stato)
        .reset_index(name="Righe")
    )




# %% [16]


# ==============================================================
# ESPORTAZIONE
# ==============================================================
# Scrive su OneLake un unico file Excel con un foglio per tabella.
#
# Tabelle disponibili:
#
#     df_fatture               lato carichi
#     df_xml_fatture           lato XML, una riga per linea
#     df_xml_codici_articolo   lato XML, una riga per codice
#     df_xml_righe             lato XML, una riga per articolo
#     map_ddt                  abbinamento dei DDT fra i due lati
#     df_merge                 full outer, il risultato del processo
#
# df_merge è la tabella da consegnare. Le altre servono a
# risalire ai passaggi quando un abbinamento non torna.
# ==============================================================

CARTELLA_EXPORT = f"{cartella}/export"
FILE_EXPORT = f"{CARTELLA_EXPORT}/riconciliazione.xlsx"

# Tabelle da esportare: nome del foglio → DataFrame.
# Per esportare soltanto il risultato:
#
#     TABELLE_DA_ESPORTARE = {
#         "df_merge": df_merge,
#     }
#
TABELLE_DA_ESPORTARE = {
    "df_merge": df_merge,
    "map_ddt": map_ddt,
    "df_xml_righe": df_xml_righe,
    "df_fatture": df_fatture,
    "df_xml_fatture": df_xml_fatture,
    "df_xml_codici_articolo": df_xml_codici_articolo,
}

with pd.ExcelWriter(FILE_EXPORT, engine="openpyxl") as writer:

    for nome_foglio, tabella in TABELLE_DA_ESPORTARE.items():

        if tabella is None or tabella.empty:
            print(f"Nessun dato da esportare per: {nome_foglio}")
            continue

        # Excel accetta nomi di foglio fino a 31 caratteri
        tabella.to_excel(
            writer,
            sheet_name=nome_foglio[:31],
            index=False,
        )

        print(
            f"Esportato {nome_foglio}: "
            f"{len(tabella):,} righe, {len(tabella.columns)} colonne"
        )

print("File salvato:", FILE_EXPORT)


ZIP copiato in locale: /tmp/xml.zip
XML trovati nello ZIP: 153


XML presenti nello ZIP: 153
XML FPR12 conservati in memoria: 153


Righe delle fatture XML:


Codici articolo XML:


File XML presenti in memoria: 153
Righe XML estratte: 9405
Codici articolo XML estratti: 14541

Distribuzione abbinamenti DDT:


Fogli presenti nel file: ['RegoleCodiciXML', 'RegoleDDT']
Foglio non presente e quindi non considerato: RegoleCodiciCarico
Righe lette / righe con CFOR per foglio:
  RegoleCodiciXML: 20 righe lette, 16 con CFOR
  RegoleCodiciCarico: foglio assente
  RegoleDDT: 4 righe lette, 4 con CFOR

Righe attive nel dizionario codici XML: 16
Righe attive nel dizionario codici carico: 0
Righe attive nel dizionario DDT: 4

Regole non riconosciute:


... (15 outputs left out)

Esportato df_merge: 9,689 righe, 71 colonne
Esportato map_ddt: 517 righe, 10 colonne
Esportato df_xml_righe: 8,418 righe, 21 colonne


Esportato df_fatture: 7,869 righe, 26 colonne
Esportato df_xml_fatture: 9,405 righe, 21 colonne


Esportato df_xml_codici_articolo: 14,541 righe, 7 colonne


File salvato: abfss://d072e408-7302-41de-a976-f6b6b626177e@onelake.dfs.fabric.microsoft.com/401c1a6a-fa86-43ab-a153-01e4fa9de79f/Files/export/riconciliazione.xlsx


In [14]:
# %% [16]

# ==============================================================
# EXPORT POWER BI - DF_MERGE RIDOTTO E CONSOLIDATO
# ==============================================================
# Crea un nuovo DataFrame destinato a Power BI contenente:
#
#   - Fattura
#   - Data fattura
#   - CEAS
#   - CFOR
#   - Denominazione
#   - Bolla
#   - Data bolla
#   - Descrizione
#   - Numero riga fattura
#   - Unita di misura
#   - Codice articolo interno
#   - Codice articolo fornitore
#   - Codice EAN
#   - Quantita XML
#   - Quantita carico
#   - Differenza quantita
#   - Prezzo unitario XML
#   - Prezzo unitario carico
#   - Totale XML
#   - Totale Mersy
#   - Differenza totali
#   - CPCO
#   - Stato DDT
#   - CPCO descrizione
#   - Stato abbinamento
#
# Il DataFrame originale df_merge non viene modificato.
#
# Output:
#     Files/export/riconciliazione_powerbi.xlsx
#
# Il file Excel contiene un solo foglio:
#     df_merge
# ==============================================================

import pandas as pd


# --------------------------------------------------------------
# 1. CONTROLLI INIZIALI
# --------------------------------------------------------------

if "df_merge" not in globals():
    raise NameError(
        "df_merge non esiste. "
        "Eseguire prima la cella [15]."
    )

if df_merge is None or df_merge.empty:
    raise ValueError(
        "df_merge e' vuoto: non ci sono dati da esportare."
    )


# --------------------------------------------------------------
# 2. COPIA DEL DATAFRAME ORIGINALE
# --------------------------------------------------------------

df_sorgente_powerbi = df_merge.copy()


# --------------------------------------------------------------
# 3. FUNZIONI DI SUPPORTO
# --------------------------------------------------------------

def pulisci_testo(serie):
    """
    Considera mancanti anche le stringhe vuote
    o contenenti soltanto spazi.

    La funzione viene utilizzata soltanto per preparare
    correttamente i valori usati dal coalesce.
    """

    return (
        serie
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )


def coalesce_colonne(
    df,
    colonne,
    default=pd.NA,
    ignora_colonne_assenti=False,
):
    """
    Restituisce il primo valore valido da sinistra verso destra.

    Equivale concettualmente al COALESCE di DAX.

    Parametri
    ----------
    df:
        DataFrame sorgente.

    colonne:
        Elenco ordinato delle colonne da utilizzare.
        La prima colonna ha la priorita maggiore.

    default:
        Valore utilizzato quando tutte le colonne sono vuote.

    ignora_colonne_assenti:
        Se True, le colonne non presenti nel DataFrame vengono
        ignorate. Questa opzione viene usata per le colonne XML
        opzionali, come DataDDT.
    """

    colonne_presenti = [
        colonna
        for colonna in colonne
        if colonna in df.columns
    ]

    colonne_mancanti = [
        colonna
        for colonna in colonne
        if colonna not in df.columns
    ]

    if colonne_mancanti and not ignora_colonne_assenti:
        raise KeyError(
            "Colonne mancanti in df_merge: "
            + ", ".join(colonne_mancanti)
        )

    if not colonne_presenti:
        risultato = pd.Series(
            pd.NA,
            index=df.index,
        )

        if default is not pd.NA:
            risultato = risultato.fillna(default)

        return risultato

    risultato = df[colonne_presenti[0]].copy()

    if (
        pd.api.types.is_object_dtype(risultato.dtype)
        or pd.api.types.is_string_dtype(risultato.dtype)
    ):
        risultato = pulisci_testo(risultato)

    for colonna in colonne_presenti[1:]:
        serie_successiva = df[colonna].copy()

        if (
            pd.api.types.is_object_dtype(
                serie_successiva.dtype
            )
            or pd.api.types.is_string_dtype(
                serie_successiva.dtype
            )
        ):
            serie_successiva = pulisci_testo(
                serie_successiva
            )

        risultato = risultato.combine_first(
            serie_successiva
        )

    if default is not pd.NA:
        risultato = risultato.fillna(default)

    return risultato


# --------------------------------------------------------------
# 4. FORNITORE CONSOLIDATO
# --------------------------------------------------------------
# Priorita:
#
#   1. CFOR proveniente dall'XML
#   2. CFOR proveniente da Mersy
# --------------------------------------------------------------

df_sorgente_powerbi["Fornitore_consolidato"] = (
    coalesce_colonne(
        df_sorgente_powerbi,
        [
            "CFOR",
            "mersy_fornitore(CFOR)",
        ],
    )
)


# --------------------------------------------------------------
# 5. DENOMINAZIONE CONSOLIDATA
# --------------------------------------------------------------
# Se Denominazione e' presente in df_merge, viene costruita
# una mappa:
#
#     CFOR -> Denominazione
#
# Questo permette di propagare la denominazione anche sulle
# righe Solo carico appartenenti allo stesso CFOR.
#
# Esempio:
#
#     Granarolo (12345)
#
# Se Denominazione non e' presente, viene usato soltanto:
#
#     12345
#
# NOTA - righe senza CFOR (fatture fuori perimetro)
# Il CFOR si ricava dai carichi (cella [11]): una fattura presente
# negli XML ma priva di carichi resta senza CFOR, anche quando il
# suo XML riporta la denominazione. Poiche' la mappa lavora su
# CFOR, per queste righe l'etichetta finale resta
# "Fornitore non disponibile".
#
# Oggi il caso e' considerato non significativo: si assume che ogni
# fattura abbia almeno un carico. Se in futuro non fosse piu' vero,
# le strade sono, in ordine di preferenza:
#
#   1. colonna separata (es. "Denominazione XML") valorizzata con
#      la Denominazione della riga stessa: non tocca l'etichetta ne'
#      i raggruppamenti del report;
#   2. fallback sull'etichetta, con
#      denominazione_propagata.fillna(df["Denominazione"]):
#      lo stesso fornitore finirebbe pero' in due gruppi nelle
#      pivot per Denominazione ("Nome (CFOR)" e "Nome");
#   3. ricavare il CFOR dall'XML invece che dai carichi, tramite la
#      partita IVA di CedentePrestatore e un'anagrafica
#      PartitaIVA -> CFOR.
#
# Qualunque strada si scelga, il conteggio delle fatture fuori
# perimetro e' gia' stampato dalla cella [8]
# ("Righe escluse perche' senza fattura XML").
# --------------------------------------------------------------

if "Denominazione" in df_sorgente_powerbi.columns:

    df_sorgente_powerbi["Denominazione_pulita"] = (
        pulisci_testo(
            df_sorgente_powerbi["Denominazione"]
        )
    )

    righe_con_denominazione = (
        df_sorgente_powerbi.loc[
            df_sorgente_powerbi[
                "Fornitore_consolidato"
            ].notna()
            & df_sorgente_powerbi[
                "Denominazione_pulita"
            ].notna(),
            [
                "Fornitore_consolidato",
                "Denominazione_pulita",
            ],
        ]
        .drop_duplicates()
    )

    if not righe_con_denominazione.empty:

        mappa_denominazioni = (
            righe_con_denominazione
            .sort_values(
                [
                    "Fornitore_consolidato",
                    "Denominazione_pulita",
                ]
            )
            .drop_duplicates(
                subset=["Fornitore_consolidato"],
                keep="first",
            )
            .set_index("Fornitore_consolidato")[
                "Denominazione_pulita"
            ]
        )

        denominazione_propagata = (
            df_sorgente_powerbi[
                "Fornitore_consolidato"
            ]
            .map(mappa_denominazioni)
        )

    else:

        denominazione_propagata = pd.Series(
            pd.NA,
            index=df_sorgente_powerbi.index,
        )

else:

    print(
        "ATTENZIONE: la colonna 'Denominazione' non e' "
        "presente in df_merge. Verra' utilizzato solo il CFOR."
    )

    denominazione_propagata = pd.Series(
        pd.NA,
        index=df_sorgente_powerbi.index,
    )


def crea_etichetta_fornitore(
    denominazione,
    cfor,
):
    """
    Costruisce l'etichetta finale del fornitore.

    Esempi:
        Granarolo + 12345 -> (12345) Granarolo
        nessun nome + 12345 -> (12345)
        Granarolo + nessun CFOR -> Granarolo
        entrambi assenti -> Fornitore non disponibile
    """

    denominazione_valida = (
        denominazione is not None
        and not pd.isna(denominazione)
        and str(denominazione).strip() != ""
    )

    cfor_valido = (
        cfor is not None
        and not pd.isna(cfor)
        and str(cfor).strip() != ""
    )

    if denominazione_valida and cfor_valido:
        return (
            f"({str(cfor).strip()}) "
            f"{str(denominazione).strip()}"
        )

    if denominazione_valida:
        return str(denominazione).strip()

    if cfor_valido:
        return str(cfor).strip()

    return "Fornitore non disponibile"


denominazione_finale = [
    crea_etichetta_fornitore(
        denominazione,
        cfor,
    )
    for denominazione, cfor in zip(
        denominazione_propagata,
        df_sorgente_powerbi["Fornitore_consolidato"],
    )
]


# --------------------------------------------------------------
# 6. CREAZIONE DEL DATAFRAME POWER BI
# --------------------------------------------------------------

df_powerbi = pd.DataFrame(
    index=df_sorgente_powerbi.index
)


# --------------------------------------------------------------
# FATTURA
# --------------------------------------------------------------
# Prima il numero fattura XML, poi quello Mersy.
# --------------------------------------------------------------

df_powerbi["Fattura"] = coalesce_colonne(
    df_sorgente_powerbi,
    [
        "NumeroFattura",
        "mersy_n_fattura(NDOGIAA)",
    ],
    default="Senza fattura",
)


# --------------------------------------------------------------
# DATA FATTURA
# --------------------------------------------------------------
# Prima la data XML, poi quella Mersy.
# --------------------------------------------------------------

df_powerbi["Data fattura"] = coalesce_colonne(
    df_sorgente_powerbi,
    [
        "DataFattura",
        "mersy_data_fattura(TDATIAA)",
    ],
)


# --------------------------------------------------------------
# TIPOLOGIA DOCUMENTO
# --------------------------------------------------------------
# 60I = fattura, 60C = nota di credito (e in futuro altre sigle).
#
# La colonna arriva gia' risolta dalla cella [15]: viene dal carico
# quando la riga ce l'ha, altrimenti dal carico della stessa
# fattura. Sulle fatture che nei carichi non esistono resta vuota:
# il tipo SDI dell'XML (TD01, TD04, ...) e' un codice diverso e non
# e' mappato su CDOCIAA.
# --------------------------------------------------------------

df_powerbi["Tipologia documento"] = (
    df_sorgente_powerbi["TipologiaDocumento"]
)


# --------------------------------------------------------------
# CEAS
# --------------------------------------------------------------
# Solo dal lato carichi: e' l'archiviazione ottica registrata dal
# gestionale. Le righe Solo XML restano vuote, perche' nell'XML
# il CEAS non e' un dato del documento ma coincide con il nome
# del file.
# --------------------------------------------------------------

df_powerbi["CEAS"] = coalesce_colonne(
    df_sorgente_powerbi,
    [
        "mersy_archiviazione_ottica(CEAS)",
    ],
)


# --------------------------------------------------------------
# BOLLA / DDT
# --------------------------------------------------------------
# NumeroDDT_Comune e' gia' consolidato nella cella [15].
# --------------------------------------------------------------

df_powerbi["Bolla"] = coalesce_colonne(
    df_sorgente_powerbi,
    [
        "NumeroDDT_Comune",
    ],
    default="Senza DDT",
)


# --------------------------------------------------------------
# DATA BOLLA
# --------------------------------------------------------------
# Priorita:
#
#   1. DataDDT XML, se disponibile
#   2. DataDDT_XML derivata dalla mappa DDT, se disponibile
#   3. Data DDT Mersy
#
# Le colonne XML mancanti vengono ignorate.
# --------------------------------------------------------------

df_powerbi["Data bolla"] = coalesce_colonne(
    df_sorgente_powerbi,
    [
        "DataDDT",
        "DataDDT_XML",
        "mersy_data_DDT(TDATIAB)",
    ],
    ignora_colonne_assenti=True,
)


# --------------------------------------------------------------
# DESCRIZIONE
# --------------------------------------------------------------
# Prima la descrizione XML, poi quella Mersy.
# --------------------------------------------------------------

df_powerbi["Descrizione"] = coalesce_colonne(
    df_sorgente_powerbi,
    [
        "Descrizione",
        "mersy_descrizione_articolo(XART)",
    ],
    default="Descrizione non disponibile",
)


# --------------------------------------------------------------
# NUMERO RIGA FATTURA
# --------------------------------------------------------------
# Solo NumeriLineaXML: sulle righe senza lato XML la colonna
# resta vuota. Non si ripiega sullo stato di abbinamento, che
# non e' un numero di riga e falserebbe i conteggi.
# --------------------------------------------------------------

df_powerbi["Numero riga fattura"] = coalesce_colonne(
    df_sorgente_powerbi,
    [
        "NumeriLineaXML",
    ],
)


# --------------------------------------------------------------
# UNITA DI MISURA
# --------------------------------------------------------------
# Prima il valore XML, poi quello Mersy.
# --------------------------------------------------------------

df_powerbi["Unità di misura"] = coalesce_colonne(
    df_sorgente_powerbi,
    [
        "UnitaMisura",
        "mersy_um_prezzo(CUMF)",
    ],
)


# --------------------------------------------------------------
# CODICE ARTICOLO INTERNO
# --------------------------------------------------------------
# Prima il codice cliente XML, poi CART Mersy.
# --------------------------------------------------------------

df_powerbi["Codice articolo interno"] = (
    coalesce_colonne(
        df_sorgente_powerbi,
        [
            "Codice Art. Cliente",
            "mersy_articolo(CART)",
        ],
    )
)


# --------------------------------------------------------------
# CODICE ARTICOLO FORNITORE
# --------------------------------------------------------------
# Prima il codice fornitore XML, poi CALA/CARF Mersy.
# --------------------------------------------------------------

df_powerbi["Codice articolo fornitore"] = (
    coalesce_colonne(
        df_sorgente_powerbi,
        [
            "Codice Art. Fornitore",
            "mersy_articolo_fornitore(CALA)",
        ],
    )
)


# --------------------------------------------------------------
# CODICE EAN
# --------------------------------------------------------------
# Prima l'EAN XML, poi l'EAN Mersy.
# --------------------------------------------------------------

df_powerbi["Codice EAN"] = coalesce_colonne(
    df_sorgente_powerbi,
    [
        "EAN",
        "mersy_ean(CEAN)",
    ],
)


# --------------------------------------------------------------
# QUANTITA
# --------------------------------------------------------------

df_powerbi["Quantità XML"] = (
    df_sorgente_powerbi["Quantita"]
)

df_powerbi["Quantità carico"] = (
    df_sorgente_powerbi[
        "mersy_qta_fattura(Qtà)"
    ]
)

# La differenza si calcola sui VALORI ASSOLUTI: una nota di
# credito ha quantità negative, e sottraendo i valori con segno una
# riga 6 contro -6 darebbe 12 invece di 0. Il segno non è un
# disallineamento, è il tipo di documento (vedi Tipologia
# documento), quindi qui va neutralizzato.
#
# Le due colonne "Quantità XML" e "Quantità carico" restano col
# segno originale: cambia solo il calcolo della differenza.
df_powerbi["Differenza quantità"] = (
    df_powerbi["Quantità XML"].abs()
    - df_powerbi["Quantità carico"].abs()
)


# --------------------------------------------------------------
# PREZZI UNITARI
# --------------------------------------------------------------
# Prezzo unitario XML:
# PrezzoUnitarioNetto viene calcolato nella cella [13]
# come PrezzoTotale / Quantita.
#
# Prezzo unitario carico:
# PF02IAL proveniente da Mersy.
# --------------------------------------------------------------

df_powerbi["Prezzo unitario XML"] = (
    df_sorgente_powerbi["PrezzoUnitarioNetto"]
)

df_powerbi["Prezzo unitario carico"] = (
    df_sorgente_powerbi[
        "mersy_prezzo_fattura(PF02IAL)"
    ]
)


# --------------------------------------------------------------
# TOTALI
# --------------------------------------------------------------

df_powerbi["Totale XML"] = (
    df_sorgente_powerbi["PrezzoTotale"]
)

df_powerbi["Totale Mersy"] = (
    df_sorgente_powerbi[
        "mersy_importo_ns_accettazione(Importo_ns_fattura)"
    ]
)

df_powerbi["Differenza totali"] = (
    df_powerbi["Totale XML"]
    - df_powerbi["Totale Mersy"]
)


# --------------------------------------------------------------
# CODICE FORNITORE
# --------------------------------------------------------------
# E' il CFOR consolidato gia' calcolato sopra: prima quello
# dell'XML, poi quello dei carichi. Resta vuoto solo quando la
# riga non ha nessuno dei due.
# --------------------------------------------------------------

df_powerbi["CFOR"] = (
    df_sorgente_powerbi["Fornitore_consolidato"]
)


# --------------------------------------------------------------
# PUNTO DI CONSEGNA
# --------------------------------------------------------------
# Valorizzato solo sulle righe che hanno un carico: una bolla
# non ancora consegnata non ha punto di consegna, ed e' corretto
# che la colonna resti vuota.
# --------------------------------------------------------------

df_powerbi["CPCO"] = (
    df_sorgente_powerbi["mersy_punto_consegna(CPCO)"]
)

df_powerbi["CPCO descrizione"] = coalesce_colonne(
    df_sorgente_powerbi,
    [
        "mersy_descrizione_punto_consegna(XPCO)",
    ],
)


# --------------------------------------------------------------
# DENOMINAZIONE
# --------------------------------------------------------------

df_powerbi["Denominazione"] = (
    denominazione_finale
)


# --------------------------------------------------------------
# STATO DDT
# --------------------------------------------------------------
# Stato dell'abbinamento della bolla, non della riga: tutte le
# righe della stessa bolla riportano lo stesso valore. La
# colonna arriva già calcolata dalla cella [15] (StatoDDT) ed è
# in italiano, perché è letta da chi riconcilia.
#
# Valori possibili:
#
#   Bolla abbinata per numero
#   Bolla abbinata per data
#   Bolla abbinata per esclusione
#   Bolla abbinata per esclusione, data mancante
#   Bolla dedotta dal carico
#   Bolla ambigua: più bolle nel carico
#   Bolla XML senza riscontro nei carichi
#   Bolla carico senza riscontro nell'XML
#   Nessuna bolla
# --------------------------------------------------------------

df_powerbi["Stato DDT"] = (
    df_sorgente_powerbi["StatoDDT"]
)


# --------------------------------------------------------------
# STATO ABBINAMENTO
# --------------------------------------------------------------
# Valori possibili:
#
#   Abbinato
#   Solo XML
#   Solo carico
#   Da verificare
# --------------------------------------------------------------

df_powerbi["Stato abbinamento"] = (
    df_sorgente_powerbi["StatoAbbinamento"]
)


# --------------------------------------------------------------
# 7. ORDINE FINALE DELLE COLONNE
# --------------------------------------------------------------

COLONNE_POWERBI = [
    "Fattura",
    "Data fattura",
    "Tipologia documento",
    "CEAS",
    "CFOR",
    "Denominazione",
    "Bolla",
    "Data bolla",
    "Stato DDT",
    "CPCO",
    "CPCO descrizione",
    "Descrizione",
    "Numero riga fattura",
    "Unità di misura",
    "Codice articolo interno",
    "Codice articolo fornitore",
    "Codice EAN",
    "Quantità XML",
    "Quantità carico",
    "Differenza quantità",
    "Prezzo unitario XML",
    "Prezzo unitario carico",
    "Totale XML",
    "Totale Mersy",
    "Differenza totali",
    "Stato abbinamento",
]

df_powerbi = (
    df_powerbi[COLONNE_POWERBI]
    .reset_index(drop=True)
)


# --------------------------------------------------------------
# 8. DIAGNOSTICA
# --------------------------------------------------------------

print(
    f"Dataset Power BI creato: "
    f"{len(df_powerbi):,} righe, "
    f"{len(df_powerbi.columns):,} colonne"
)

print()
print("Anteprima dataset Power BI:")

display(
    df_powerbi.head(20)
)


print()
print("Distribuzione per Tipologia documento:")

display(
    df_powerbi["Tipologia documento"]
    .value_counts(dropna=False)
    .rename_axis("Tipologia documento")
    .reset_index(name="Righe")
)

print()
print("Distribuzione per Stato DDT:")

display(
    df_powerbi["Stato DDT"]
    .value_counts(dropna=False)
    .rename_axis("Stato DDT")
    .reset_index(name="Righe")
)

print()
print("Distribuzione per Stato abbinamento:")

display(
    df_powerbi["Stato abbinamento"]
    .value_counts(dropna=False)
    .rename_axis("Stato abbinamento")
    .reset_index(name="Numero righe")
)


# --------------------------------------------------------------
# 9. ESPORTAZIONE EXCEL
# --------------------------------------------------------------
# Esporta soltanto df_powerbi.
#
# Il file Excel conterra' un unico foglio:
#
#     df_merge
#
# Non vengono esportati map_ddt, df_fatture o gli altri
# DataFrame diagnostici.
# --------------------------------------------------------------

CARTELLA_EXPORT = f"{cartella}/export"

FILE_EXPORT_POWERBI = (
    f"{CARTELLA_EXPORT}/riconciliazione_powerbi.xlsx"
)


if not notebookutils.fs.exists(CARTELLA_EXPORT):
    notebookutils.fs.mkdirs(CARTELLA_EXPORT)


with pd.ExcelWriter(
    FILE_EXPORT_POWERBI,
    engine="openpyxl",
    date_format="DD/MM/YYYY",
    datetime_format="DD/MM/YYYY",
) as writer:

    df_powerbi.to_excel(
        writer,
        sheet_name="df_merge",
        index=False,
    )


# --------------------------------------------------------------
# 10. RIEPILOGO FINALE
# --------------------------------------------------------------

print()
print(
    "File Excel Power BI esportato: "
    f"{len(df_powerbi):,} righe, "
    f"{len(df_powerbi.columns):,} colonne"
)

print("Foglio esportato: df_merge")
print("File salvato:", FILE_EXPORT_POWERBI)

ATTENZIONE: la colonna 'Denominazione' non e' presente in df_merge. Verra' utilizzato solo il CFOR.
Dataset Power BI creato: 9,689 righe, 20 colonne

Anteprima dataset Power BI:



Distribuzione per Stato abbinamento:



File Excel Power BI esportato: 9,689 righe, 20 colonne
Foglio esportato: df_merge
File salvato: abfss://d072e408-7302-41de-a976-f6b6b626177e@onelake.dfs.fabric.microsoft.com/401c1a6a-fa86-43ab-a153-01e4fa9de79f/Files/export/riconciliazione_powerbi.xlsx
